In [ ]:
import sys
import platform
from importlib import metadata, util

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())

for package, module in [
    ("torch", "torch"),
    ("numpy", "numpy"),
    ("scipy", "scipy"),
    ("pandas", "pandas"),
    ("scikit-learn", "sklearn"),
]:
    if util.find_spec(module) is None:
        print(f"{package}: NOT AVAILABLE")
    else:
        try:
            print(f"{package}: {metadata.version(package)}")
        except metadata.PackageNotFoundError:
            print(f"{package}: available, version unknown")

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

In [ ]:
from itertools import permutations, product
from collections import Counter
import json


def compose(left, right):
    return tuple(left[right[index]] for index in range(3))


def inverse(state):
    return tuple(state.index(index) for index in range(3))


def parity(state):
    return sum(
        state[first] > state[second]
        for first in range(3)
        for second in range(first + 1, 3)
    ) % 2


def canonical(blocks):
    return sorted([sorted(block) for block in blocks])


def q8_multiply(left, right):
    left_sign, left_basis = left
    right_sign, right_basis = right

    if left_basis == 0:
        return (left_sign * right_sign, right_basis)
    if right_basis == 0:
        return (left_sign * right_sign, left_basis)
    if left_basis == right_basis:
        return (-left_sign * right_sign, 0)

    positive_pairs = {
        (1, 2): 3,
        (2, 3): 1,
        (3, 1): 2,
    }

    if (left_basis, right_basis) in positive_pairs:
        return (
            left_sign * right_sign,
            positive_pairs[(left_basis, right_basis)],
        )

    return (
        -left_sign * right_sign,
        positive_pairs[(right_basis, left_basis)],
    )


def d4_multiply(left, right):
    left_rotation, left_reflection = left
    right_rotation, right_reflection = right

    return (
        (left_rotation + (-1) ** left_reflection * right_rotation) % 4,
        (left_reflection + right_reflection) % 2,
    )


def left_orbits(states, subgroup, multiply):
    blocks = {
        frozenset(multiply(member, state) for member in subgroup)
        for state in states
    }
    return canonical(blocks)


def validate_group(states, identity, multiply, actions):
    assert len(set(states)) == len(states), "Duplicate states"

    assert all(
        multiply(left, right) in states
        for left in states
        for right in states
    ), "Closure failed"

    assert all(
        multiply(identity, state) == state
        and multiply(state, identity) == state
        for state in states
    ), "Identity failed"

    assert all(
        any(
            multiply(state, candidate) == identity
            and multiply(candidate, state) == identity
            for candidate in states
        )
        for state in states
    ), "Inverse failed"

    assert all(
        multiply(multiply(first, second), third)
        == multiply(first, multiply(second, third))
        for first in states
        for second in states
        for third in states
    ), "Associativity failed"

    assert any(
        multiply(left, right) != multiply(right, left)
        for left in states
        for right in states
    ), "Expected a noncommutative group"

    reached = {identity}

    while True:
        expanded = reached | {
            multiply(state, action)
            for state in reached
            for action in actions
        }
        if expanded == reached:
            break
        reached = expanded

    assert reached == set(states), "Actions do not generate the full group"


def audit(states, multiply, actions, partition):
    flattened = [state for block in partition for state in block]

    assert (
        len(flattened) == len(states)
        and set(flattened) == set(states)
    ), "Invalid partition"

    labels = {
        state: index
        for index, block in enumerate(partition)
        for state in block
    }

    violations = []
    correct = 0

    for block in partition:
        for action in actions:
            destinations = Counter(
                labels[multiply(state, action)]
                for state in block
            )
            correct += max(destinations.values())

            if len(destinations) > 1:
                violations.append({
                    "states": block,
                    "action": action,
                    "next_blocks": [
                        labels[multiply(state, action)]
                        for state in block
                    ],
                })

    refined = canonical(partition)
    sizes = [len(refined)]

    while True:
        refined_labels = {
            state: index
            for index, block in enumerate(refined)
            for state in block
        }
        buckets = {}

        for state in states:
            signature = (
                refined_labels[state],
                tuple(
                    refined_labels[multiply(state, action)]
                    for action in actions
                ),
            )
            buckets.setdefault(signature, []).append(state)

        updated = canonical(buckets.values())

        if updated == refined:
            break

        refined = updated
        sizes.append(len(refined))

    return {
        "initial_blocks": len(partition),
        "compatible": not violations,
        "next_label_accuracy_ceiling": (
            correct / (len(states) * len(actions))
        ),
        "refinement_block_counts": sizes,
        "first_violation": violations[0] if violations else None,
    }


s3_states = list(permutations(range(3)))
q8_states = list(product((-1, 1), range(4)))
d4_states = list(product(range(4), range(2)))

s3_actions = [(1, 0, 2), (0, 2, 1)]
q8_actions = [(1, 1), (1, 2)]
d4_actions = [(1, 0), (0, 1)]

validate_group(s3_states, (0, 1, 2), compose, s3_actions)
validate_group(q8_states, (1, 0), q8_multiply, q8_actions)
validate_group(d4_states, (0, 0), d4_multiply, d4_actions)

s3_conjugacy = canonical({
    frozenset(
        compose(compose(change, state), inverse(change))
        for change in s3_states
    )
    for state in s3_states
})

experiments = {
    "S3_conjugacy": (
        s3_states,
        compose,
        s3_actions,
        s3_conjugacy,
    ),
    "S3_A3": (
        s3_states,
        compose,
        s3_actions,
        [
            [state for state in s3_states if parity(state) == value]
            for value in range(2)
        ],
    ),
    "Q8_center": (
        q8_states,
        q8_multiply,
        q8_actions,
        left_orbits(
            q8_states,
            [(1, 0), (-1, 0)],
            q8_multiply,
        ),
    ),
    "D4_center": (
        d4_states,
        d4_multiply,
        d4_actions,
        left_orbits(
            d4_states,
            [(0, 0), (2, 0)],
            d4_multiply,
        ),
    ),
}

report = {
    name: audit(*arguments)
    for name, arguments in experiments.items()
}

assert not report["S3_conjugacy"]["compatible"]
assert report["S3_conjugacy"]["refinement_block_counts"] == [3, 5, 6]

assert all(
    report[name]["compatible"]
    for name in ["S3_A3", "Q8_center", "D4_center"]
)

assert report["S3_A3"]["initial_blocks"] == 2
assert report["Q8_center"]["initial_blocks"] == 4
assert report["D4_center"]["initial_blocks"] == 4

print("PASS: group axioms, generator coverage, quotients, and S3 counterexample.")
print(json.dumps(report, indent=2))

## 1. Paired Q8 dataset

This pilot uses a newly specified dataset, not a reconstruction of the
legacy experiments.

Each sequence contains an initial group element followed by 12 actions.
The actions are right multiplications by i, j, -i, or -j.

Each pair starts from g and -g and receives the same action sequence.
The paired trajectories have identical quotient states and opposite
central-sign labels at every position.

### Encoding

- Initial-state tokens 0–7: 1, -1, i, -i, j, -j, k, -k.
- Action tokens 8–11: i, j, -i, -j.
- Full-state label: 2 × basis + negative-sign indicator.
- Quotient-state label: basis, taking values 0–3.

Targets describe the state after processing the current token.

### Splits and leakage controls

- Training: 4,096 pairs / 8,192 sequences.
- Validation: 512 pairs / 1,024 sequences.
- Test: 512 pairs / 1,024 sequences.

Paired trajectories remain in the same split. Initial quotient and complete
action sequence jointly identify a split unit; these units do not overlap
between splits.

Central-sign labels are exactly balanced. Quotient labels are approximately
balanced.

**Status:** Pairing and split-disjointness checks passed.

In [ ]:
from pathlib import Path
import json
import numpy as np

DATA_SEED = 101
SEQUENCE_LENGTH = 12

OUTPUT_DIR = Path("q8_pilot_v1")
OUTPUT_DIR.mkdir(exist_ok=True)

ACTION_ELEMENTS = [
    (1, 1),
    (1, 2),
    (-1, 1),
    (-1, 2),
]


def q8_step(left, right):
    left_sign, left_basis = left
    right_sign, right_basis = right

    if left_basis == 0:
        return left_sign * right_sign, right_basis
    if right_basis == 0:
        return left_sign * right_sign, left_basis
    if left_basis == right_basis:
        return -left_sign * right_sign, 0

    positive = {
        (1, 2): 3,
        (2, 3): 1,
        (3, 1): 2,
    }

    if (left_basis, right_basis) in positive:
        return (
            left_sign * right_sign,
            positive[(left_basis, right_basis)],
        )

    return (
        -left_sign * right_sign,
        positive[(right_basis, left_basis)],
    )


def encode_q8(state):
    sign, basis = state
    return 2 * basis + int(sign < 0)


def make_split(pair_count, length, rng, used_keys):
    tokens = np.empty(
        (2 * pair_count, length + 1),
        dtype=np.int64,
    )
    full_state = np.empty_like(tokens)
    split_keys = []

    for pair_index in range(pair_count):
        while True:
            initial_basis = int(rng.integers(4))
            action_ids = rng.integers(4, size=length)
            key = (initial_basis, *map(int, action_ids))

            if key not in used_keys:
                used_keys.add(key)
                split_keys.append(key)
                break

        for twin_index, initial_sign in enumerate((1, -1)):
            row = 2 * pair_index + twin_index
            state = (initial_sign, initial_basis)

            tokens[row, 0] = encode_q8(state)
            tokens[row, 1:] = 8 + action_ids
            full_state[row, 0] = encode_q8(state)

            for position, action_id in enumerate(action_ids, start=1):
                state = q8_step(
                    state,
                    ACTION_ELEMENTS[int(action_id)],
                )
                full_state[row, position] = encode_q8(state)

    quotient_state = full_state // 2
    nuisance = full_state % 2

    assert np.array_equal(
        tokens[::2, 1:],
        tokens[1::2, 1:],
    )
    assert np.array_equal(
        quotient_state[::2],
        quotient_state[1::2],
    )
    assert np.array_equal(
        nuisance[::2],
        1 - nuisance[1::2],
    )

    data = {
        "tokens": tokens,
        "full_state": full_state,
        "quotient_state": quotient_state,
        "nuisance": nuisance,
        "pair_id": np.repeat(np.arange(pair_count), 2),
        "sequence_id": np.arange(2 * pair_count),
    }

    return data, set(split_keys)


rng = np.random.default_rng(DATA_SEED)
used_keys = set()
datasets = {}
split_keys = {}

split_sizes = {
    "train": 4096,
    "validation": 512,
    "test": 512,
}

for split_name, pair_count in split_sizes.items():
    data, keys = make_split(
        pair_count,
        SEQUENCE_LENGTH,
        rng,
        used_keys,
    )
    datasets[split_name] = data
    split_keys[split_name] = keys

assert not (split_keys["train"] & split_keys["validation"])
assert not (split_keys["train"] & split_keys["test"])
assert not (split_keys["validation"] & split_keys["test"])

for split_name, data in datasets.items():
    np.savez_compressed(
        OUTPUT_DIR / f"{split_name}.npz",
        **data,
    )

    quotient_counts = np.bincount(
        data["quotient_state"].ravel(),
        minlength=4,
    )
    nuisance_counts = np.bincount(
        data["nuisance"].ravel(),
        minlength=2,
    )

    print(
        f"{split_name}: "
        f"tokens={data['tokens'].shape}, "
        f"quotient_counts={quotient_counts.tolist()}, "
        f"sign_counts={nuisance_counts.tolist()}"
    )

config = {
    "experiment": "q8_pilot_v1",
    "legacy_replication": False,
    "data_seed": DATA_SEED,
    "sequence_length": SEQUENCE_LENGTH,
    "base_pairs_per_split": split_sizes,
    "state_encoding": [
        "1", "-1", "i", "-i", "j", "-j", "k", "-k"
    ],
    "action_token_encoding": {
        "8": "i",
        "9": "j",
        "10": "-i",
        "11": "-j",
    },
    "dynamics": "state_next = state * action",
    "target_alignment": "state after processing the current token",
    "split_key": "initial quotient plus complete action sequence",
    "pair_id_scope": "within split",
    "sequence_id_scope": "within split",
}

with (OUTPUT_DIR / "data_config.json").open(
    "w", encoding="utf-8"
) as config_file:
    json.dump(config, config_file, indent=2)

print("\nPASS: paired data and disjoint trajectories.")
print("Saved to:", OUTPUT_DIR)

## 2. Causal Transformer and pre-training checks

We use a small, newly specified causal Transformer:

- Hidden dimension: 64.
- Attention heads: 4.
- Transformer blocks: 3.
- Feed-forward dimension: 128.
- Pre-layer normalization and sinusoidal positional encoding.
- Dropout: 0.
- Separate full-state and quotient-state prediction heads.

The model receives only the initial-state token and action tokens.
Ground-truth state labels are used as training targets, not as model inputs.

Position 0 contains the explicitly supplied initial state.
Training and reported prediction metrics will exclude this position.

### Checks before training

1. Output dimensions match the eight full states and four quotient states.
2. Changing future tokens does not change earlier logits or hidden states.
3. Running a prefix alone reproduces its outputs in the complete sequence.
4. Disabling the causal mask exposes future-token influence, serving as a
   positive control for the leakage test.
5. Both prediction heads receive finite, nonzero training gradients.

These checks validate implementation behavior, not learned task performance.
This stage runs on CPU; GPU training has not started.

In [ ]:
import json
import math
import random
from pathlib import Path

import numpy as np
import torch
from torch import nn
import torch.nn.functional as F

MODEL_SEED = 11

random.seed(MODEL_SEED)
np.random.seed(MODEL_SEED)
torch.manual_seed(MODEL_SEED)

torch.set_num_threads(min(4, torch.get_num_threads()))

MODEL_CONFIG = {
    "vocab_size": 12,
    "d_model": 64,
    "n_heads": 4,
    "n_layers": 3,
    "d_ff": 128,
    "max_tokens": 65,
    "dropout": 0.0,
}

OUTPUT_DIR = Path("q8_pilot_v1")

assert (OUTPUT_DIR / "train.npz").exists(), (
    "Run the dataset-generation cell first."
)


class CausalBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout):
        super().__init__()

        self.attention_norm = nn.LayerNorm(d_model)
        self.attention = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=n_heads,
            dropout=dropout,
            batch_first=True,
        )

        self.feedforward_norm = nn.LayerNorm(d_model)
        self.feedforward = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
        )

    def forward(self, hidden, attention_mask):
        normalized = self.attention_norm(hidden)

        attended, _ = self.attention(
            normalized,
            normalized,
            normalized,
            attn_mask=attention_mask,
            need_weights=False,
        )

        hidden = hidden + attended
        hidden = hidden + self.feedforward(
            self.feedforward_norm(hidden)
        )
        return hidden


class Q8Transformer(nn.Module):
    def __init__(
        self,
        vocab_size,
        d_model,
        n_heads,
        n_layers,
        d_ff,
        max_tokens,
        dropout,
    ):
        super().__init__()

        assert d_model % 2 == 0
        assert d_model % n_heads == 0

        self.token_embedding = nn.Embedding(vocab_size, d_model)

        positions = torch.arange(
            max_tokens, dtype=torch.float32
        ).unsqueeze(1)

        frequencies = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float32)
            * (-math.log(10000.0) / d_model)
        )

        positional_encoding = torch.zeros(max_tokens, d_model)
        positional_encoding[:, 0::2] = torch.sin(
            positions * frequencies
        )
        positional_encoding[:, 1::2] = torch.cos(
            positions * frequencies
        )

        self.register_buffer(
            "positional_encoding",
            positional_encoding,
        )

        self.blocks = nn.ModuleList([
            CausalBlock(d_model, n_heads, d_ff, dropout)
            for _ in range(n_layers)
        ])

        self.final_norm = nn.LayerNorm(d_model)
        self.full_head = nn.Linear(d_model, 8)
        self.quotient_head = nn.Linear(d_model, 4)

    def forward(self, tokens, causal=True):
        sequence_length = tokens.shape[1]

        if sequence_length > self.positional_encoding.shape[0]:
            raise ValueError("Sequence exceeds configured maximum length.")

        hidden = (
            self.token_embedding(tokens)
            + self.positional_encoding[:sequence_length].unsqueeze(0)
        )

        attention_mask = None
        if causal:
            attention_mask = torch.triu(
                torch.ones(
                    sequence_length,
                    sequence_length,
                    dtype=torch.bool,
                    device=tokens.device,
                ),
                diagonal=1,
            )

        hidden_states = []

        for block in self.blocks:
            hidden = block(hidden, attention_mask)
            hidden_states.append(hidden)

        final_hidden = self.final_norm(hidden)

        return {
            "full_logits": self.full_head(final_hidden),
            "quotient_logits": self.quotient_head(final_hidden),
            "hidden_states": tuple(hidden_states),
            "final_hidden": final_hidden,
        }


model = Q8Transformer(**MODEL_CONFIG).cpu()

with np.load(OUTPUT_DIR / "train.npz", allow_pickle=False) as data:
    sample_tokens = torch.from_numpy(data["tokens"][:8].copy())
    sample_full = torch.from_numpy(data["full_state"][:8].copy())
    sample_quotient = torch.from_numpy(
        data["quotient_state"][:8].copy()
    )

model.eval()

prefix_length = 6
changed_tokens = sample_tokens.clone()

changed_tokens[:, prefix_length:] = (
    8 + (changed_tokens[:, prefix_length:] - 8 + 1) % 4
)

with torch.no_grad():
    original = model(sample_tokens)
    changed = model(changed_tokens)
    prefix_only = model(sample_tokens[:, :prefix_length])

    assert original["full_logits"].shape == (8, 13, 8)
    assert original["quotient_logits"].shape == (8, 13, 4)
    assert len(original["hidden_states"]) == MODEL_CONFIG["n_layers"]

    checked_keys = [
        "full_logits",
        "quotient_logits",
        "final_hidden",
    ]

    largest_future_difference = 0.0
    largest_prefix_difference = 0.0

    for output_key in checked_keys:
        original_prefix = original[output_key][:, :prefix_length]

        future_difference = (
            original_prefix
            - changed[output_key][:, :prefix_length]
        ).abs().max().item()

        prefix_difference = (
            original_prefix - prefix_only[output_key]
        ).abs().max().item()

        largest_future_difference = max(
            largest_future_difference,
            future_difference,
        )
        largest_prefix_difference = max(
            largest_prefix_difference,
            prefix_difference,
        )

        assert torch.allclose(
            original_prefix,
            changed[output_key][:, :prefix_length],
            atol=2e-6,
            rtol=2e-6,
        ), f"Future leakage detected in {output_key}"

        assert torch.allclose(
            original_prefix,
            prefix_only[output_key],
            atol=2e-6,
            rtol=2e-6,
        ), f"Prefix inconsistency detected in {output_key}"

    for layer_index, original_hidden in enumerate(
        original["hidden_states"]
    ):
        assert torch.allclose(
            original_hidden[:, :prefix_length],
            changed["hidden_states"][layer_index][:, :prefix_length],
            atol=2e-6,
            rtol=2e-6,
        ), f"Future leakage detected in layer {layer_index + 1}"

        assert torch.allclose(
            original_hidden[:, :prefix_length],
            prefix_only["hidden_states"][layer_index],
            atol=2e-6,
            rtol=2e-6,
        ), f"Prefix inconsistency in layer {layer_index + 1}"

    unmasked_original = model(sample_tokens, causal=False)
    unmasked_changed = model(changed_tokens, causal=False)

    unmasked_difference = (
        unmasked_original["full_logits"][:, :prefix_length]
        - unmasked_changed["full_logits"][:, :prefix_length]
    ).abs().max().item()

    assert unmasked_difference > 1e-6, (
        "Positive control failed: the leakage test was not sensitive."
    )

model.train()
model.zero_grad(set_to_none=True)

training_output = model(sample_tokens)

full_loss = F.cross_entropy(
    training_output["full_logits"][:, 1:].reshape(-1, 8),
    sample_full[:, 1:].reshape(-1),
)

quotient_loss = F.cross_entropy(
    training_output["quotient_logits"][:, 1:].reshape(-1, 4),
    sample_quotient[:, 1:].reshape(-1),
)

loss = 0.5 * (full_loss + quotient_loss)

assert torch.isfinite(loss)
loss.backward()

for head_name in ["full_head", "quotient_head"]:
    head = getattr(model, head_name)
    gradient = head.weight.grad

    assert gradient is not None
    assert torch.isfinite(gradient).all()
    assert gradient.abs().sum().item() > 0

for parameter_name, parameter in model.named_parameters():
    assert parameter.grad is not None, (
        f"No gradient: {parameter_name}"
    )
    assert torch.isfinite(parameter.grad).all(), (
        f"Nonfinite gradient: {parameter_name}"
    )

model.zero_grad(set_to_none=True)
model.eval()

parameter_count = sum(
    parameter.numel() for parameter in model.parameters()
)

check_results = {
    "status": "passed",
    "model_seed": MODEL_SEED,
    "parameter_count": parameter_count,
    "future_token_difference": largest_future_difference,
    "prefix_difference": largest_prefix_difference,
    "unmasked_positive_control_difference": unmasked_difference,
    "initial_smoke_test_loss": loss.item(),
    "training_started": False,
    "device": "cpu",
}

with (OUTPUT_DIR / "model_config.json").open(
    "w", encoding="utf-8"
) as config_file:
    json.dump(
        {"model_seed": MODEL_SEED, **MODEL_CONFIG},
        config_file,
        indent=2,
    )

with (OUTPUT_DIR / "implementation_checks.json").open(
    "w", encoding="utf-8"
) as results_file:
    json.dump(check_results, results_file, indent=2)

print("PASS: shapes, causality, prefix consistency, and gradients.")
print(json.dumps(check_results, indent=2))

In [ ]:
import hashlib
import json
import random
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F

print("PyTorch:", torch.__version__)
print("CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is unavailable. Do not install anything yet. "
        "Send this output before continuing."
    )

assert "Q8Transformer" in globals(), (
    "Run the model-definition cell first."
)
assert "check_results" in globals(), (
    "Run the implementation checks first."
)
assert check_results["status"] == "passed"

device = torch.device("cuda:0")
print("Using:", torch.cuda.get_device_name(0))

DATA_DIR = Path("q8_pilot_v1")
RUN_DIR = DATA_DIR / "training_smoke_seed11"

if RUN_DIR.exists():
    raise RuntimeError(
        f"{RUN_DIR} already exists. "
        "Do not overwrite it; send the previous run's output."
    )

TRAIN_CONFIG = {
    "model_seed": 11,
    "shuffle_seed": 1011,
    "batch_size": 256,
    "learning_rate": 0.001,
    "weight_decay": 0.01,
    "max_epochs": 10,
    "training_time_limit_seconds": 600,
    "gradient_clip": 1.0,
    "full_loss_weight": 0.5,
    "quotient_loss_weight": 0.5,
    "exclude_initial_position": True,
    "checkpoint_selection": "minimum validation joint loss",
}

random.seed(TRAIN_CONFIG["model_seed"])
np.random.seed(TRAIN_CONFIG["model_seed"])
torch.manual_seed(TRAIN_CONFIG["model_seed"])
torch.cuda.manual_seed_all(TRAIN_CONFIG["model_seed"])

model = Q8Transformer(**MODEL_CONFIG).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=TRAIN_CONFIG["learning_rate"],
    weight_decay=TRAIN_CONFIG["weight_decay"],
)

shuffle_generator = torch.Generator()
shuffle_generator.manual_seed(TRAIN_CONFIG["shuffle_seed"])


def load_split(split_name):
    with np.load(
        DATA_DIR / f"{split_name}.npz",
        allow_pickle=False,
    ) as archive:
        return {
            name: torch.from_numpy(archive[name].copy()).to(device)
            for name in [
                "tokens",
                "full_state",
                "quotient_state",
            ]
        }


train_data = load_split("train")
validation_data = load_split("validation")

assert train_data["tokens"].shape == (8192, 13)
assert validation_data["tokens"].shape == (1024, 13)

RUN_DIR.mkdir()

with (DATA_DIR / "data_config.json").open(
    encoding="utf-8"
) as config_file:
    data_config = json.load(config_file)

provenance = {
    "training": TRAIN_CONFIG,
    "model": MODEL_CONFIG,
    "data": data_config,
    "torch_version": str(torch.__version__),
    "numpy_version": np.__version__,
    "cuda_build": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0),
    "implementation_checks": check_results,
    "data_sha256": {
        split_name: hashlib.sha256(
            (DATA_DIR / f"{split_name}.npz").read_bytes()
        ).hexdigest()
        for split_name in ["train", "validation"]
    },
}

with (RUN_DIR / "run_config.json").open(
    "w", encoding="utf-8"
) as config_file:
    json.dump(provenance, config_file, indent=2)


@torch.no_grad()
def evaluate(data):
    model.eval()

    full_loss_sum = 0.0
    quotient_loss_sum = 0.0
    full_correct = 0
    quotient_correct = 0
    full_sequences_correct = 0
    quotient_sequences_correct = 0
    token_count = 0
    sequence_count = len(data["tokens"])

    for start in range(
        0, sequence_count, TRAIN_CONFIG["batch_size"]
    ):
        stop = start + TRAIN_CONFIG["batch_size"]
        output = model(data["tokens"][start:stop])

        full_logits = output["full_logits"][:, 1:]
        quotient_logits = output["quotient_logits"][:, 1:]

        full_target = data["full_state"][start:stop, 1:]
        quotient_target = data["quotient_state"][start:stop, 1:]

        full_loss_sum += F.cross_entropy(
            full_logits.reshape(-1, 8),
            full_target.reshape(-1),
            reduction="sum",
        ).item()

        quotient_loss_sum += F.cross_entropy(
            quotient_logits.reshape(-1, 4),
            quotient_target.reshape(-1),
            reduction="sum",
        ).item()

        full_matches = full_logits.argmax(-1) == full_target
        quotient_matches = (
            quotient_logits.argmax(-1) == quotient_target
        )

        full_correct += full_matches.sum().item()
        quotient_correct += quotient_matches.sum().item()

        full_sequences_correct += (
            full_matches.all(dim=1).sum().item()
        )
        quotient_sequences_correct += (
            quotient_matches.all(dim=1).sum().item()
        )

        token_count += full_target.numel()

    full_loss = full_loss_sum / token_count
    quotient_loss = quotient_loss_sum / token_count

    return {
        "joint_loss": 0.5 * (full_loss + quotient_loss),
        "full_loss": full_loss,
        "quotient_loss": quotient_loss,
        "full_token_accuracy": full_correct / token_count,
        "quotient_token_accuracy": quotient_correct / token_count,
        "full_sequence_accuracy": (
            full_sequences_correct / sequence_count
        ),
        "quotient_sequence_accuracy": (
            quotient_sequences_correct / sequence_count
        ),
    }


def save_checkpoint(filename, epoch, validation, complete_epoch):
    checkpoint = {
        "model_state_dict": {
            name: tensor.detach().cpu()
            for name, tensor in model.state_dict().items()
        },
        "optimizer_state_dict": optimizer.state_dict(),
        "shuffle_rng_state": shuffle_generator.get_state(),
        "torch_rng_state": torch.get_rng_state(),
        "cuda_rng_states": torch.cuda.get_rng_state_all(),
        "epoch": epoch,
        "complete_epoch": complete_epoch,
        "validation": validation,
        "model_config": MODEL_CONFIG,
        "train_config": TRAIN_CONFIG,
    }
    temporary_path = RUN_DIR / f"{filename}.tmp"
    torch.save(checkpoint, temporary_path)
    temporary_path.replace(RUN_DIR / filename)


initial_validation = evaluate(validation_data)
print("\nUntrained validation:")
print(json.dumps(initial_validation, indent=2))

history = []
best_validation_loss = float("inf")
best_epoch = None
stop_reason = "max_epochs"

torch.cuda.synchronize()
training_start = time.perf_counter()

for epoch in range(1, TRAIN_CONFIG["max_epochs"] + 1):
    if (
        time.perf_counter() - training_start
        >= TRAIN_CONFIG["training_time_limit_seconds"]
    ):
        stop_reason = "time_limit"
        break

    model.train()
    epoch_start = time.perf_counter()

    ordering = torch.randperm(
        len(train_data["tokens"]),
        generator=shuffle_generator,
    ).to(device)

    accumulated_loss = 0.0
    processed_tokens = 0
    processed_sequences = 0

    for start in range(
        0, len(ordering), TRAIN_CONFIG["batch_size"]
    ):
        if (
            time.perf_counter() - training_start
            >= TRAIN_CONFIG["training_time_limit_seconds"]
        ):
            stop_reason = "time_limit"
            break

        indices = ordering[
            start:start + TRAIN_CONFIG["batch_size"]
        ]

        optimizer.zero_grad(set_to_none=True)
        output = model(train_data["tokens"][indices])

        full_target = train_data["full_state"][indices, 1:]
        quotient_target = train_data["quotient_state"][indices, 1:]

        full_loss = F.cross_entropy(
            output["full_logits"][:, 1:].reshape(-1, 8),
            full_target.reshape(-1),
        )
        quotient_loss = F.cross_entropy(
            output["quotient_logits"][:, 1:].reshape(-1, 4),
            quotient_target.reshape(-1),
        )

        loss = 0.5 * (full_loss + quotient_loss)

        if not torch.isfinite(loss).item():
            raise RuntimeError(
                f"Nonfinite training loss at epoch {epoch}."
            )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=TRAIN_CONFIG["gradient_clip"],
            error_if_nonfinite=True,
        )
        optimizer.step()

        batch_tokens = full_target.numel()
        accumulated_loss += loss.item() * batch_tokens
        processed_tokens += batch_tokens
        processed_sequences += len(indices)

    if processed_tokens == 0:
        break

    validation = evaluate(validation_data)
    complete_epoch = processed_sequences == len(ordering)

    torch.cuda.synchronize()
    epoch_seconds = time.perf_counter() - epoch_start

    row = {
        "epoch": epoch,
        "complete_epoch": complete_epoch,
        "train_loss": accumulated_loss / processed_tokens,
        "epoch_seconds": epoch_seconds,
        **validation,
    }
    history.append(row)

    save_checkpoint(
        "last.pt", epoch, validation, complete_epoch
    )

    if validation["joint_loss"] < best_validation_loss:
        best_validation_loss = validation["joint_loss"]
        best_epoch = epoch
        save_checkpoint(
            "best.pt", epoch, validation, complete_epoch
        )

    with (RUN_DIR / "history.json").open(
        "w", encoding="utf-8"
    ) as history_file:
        json.dump(history, history_file, indent=2)

    print(
        f"epoch={epoch:02d} | "
        f"train_loss={row['train_loss']:.4f} | "
        f"val_loss={validation['joint_loss']:.4f} | "
        f"full_token={validation['full_token_accuracy']:.2%} | "
        f"quotient_token={validation['quotient_token_accuracy']:.2%} | "
        f"full_sequence={validation['full_sequence_accuracy']:.2%} | "
        f"quotient_sequence={validation['quotient_sequence_accuracy']:.2%} | "
        f"seconds={epoch_seconds:.1f} | "
        f"complete={complete_epoch}",
        flush=True,
    )

    if stop_reason == "time_limit":
        break

torch.cuda.synchronize()

summary = {
    "initial_validation": initial_validation,
    "last_validation": history[-1] if history else None,
    "best_epoch": best_epoch,
    "best_validation_loss": (
        best_validation_loss if best_epoch is not None else None
    ),
    "epochs_logged": len(history),
    "elapsed_seconds": time.perf_counter() - training_start,
    "stop_reason": stop_reason,
    "test_evaluated": False,
    "run_directory": str(RUN_DIR),
}

with (RUN_DIR / "summary.json").open(
    "w", encoding="utf-8"
) as summary_file:
    json.dump(summary, summary_file, indent=2)

print("\nPILOT FINISHED")
print(json.dumps(summary, indent=2))

In [ ]:
import json
import shutil
import time
from pathlib import Path

import torch
import torch.nn.functional as F

required_names = [
    "Q8Transformer",
    "evaluate",
    "save_checkpoint",
    "load_split",
    "shuffle_generator",
]

assert all(name in globals() for name in required_names), (
    "This cell requires the previous training session."
)
assert torch.cuda.is_available(), "GPU is unavailable."

device = torch.device("cuda:0")

PARENT_DIR = Path("q8_pilot_v1/training_smoke_seed11")
NEW_RUN_DIR = Path("q8_pilot_v1/training_continued_seed11")

assert (PARENT_DIR / "last.pt").exists(), (
    "The previous checkpoint is missing."
)
assert not NEW_RUN_DIR.exists(), (
    "Continuation results already exist. Do not overwrite them."
)

checkpoint = torch.load(
    PARENT_DIR / "last.pt",
    map_location="cpu",
    weights_only=True,
)

assert checkpoint["complete_epoch"], (
    "The checkpoint is from a partial epoch; resume needs extra handling."
)

MODEL_CONFIG = checkpoint["model_config"]
TRAIN_CONFIG = dict(checkpoint["train_config"])

starting_epoch = int(checkpoint["epoch"])

TRAIN_CONFIG.update({
    "max_epochs": starting_epoch + 100,
    "training_time_limit_seconds": 180,
    "parent_run": str(PARENT_DIR),
    "starting_epoch": starting_epoch,
    "stage": "continuation_without_architecture_or_lr_change",
})

model = Q8Transformer(**MODEL_CONFIG).to(device)
model.load_state_dict(checkpoint["model_state_dict"])

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=TRAIN_CONFIG["learning_rate"],
    weight_decay=TRAIN_CONFIG["weight_decay"],
)
optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

shuffle_generator.set_state(
    checkpoint["shuffle_rng_state"].cpu()
)
torch.set_rng_state(checkpoint["torch_rng_state"].cpu())
torch.cuda.set_rng_state_all([
    state.cpu()
    for state in checkpoint["cuda_rng_states"]
])

train_data = load_split("train")
validation_data = load_split("validation")

with (PARENT_DIR / "history.json").open(
    encoding="utf-8"
) as history_file:
    history = json.load(history_file)

previous_best = torch.load(
    PARENT_DIR / "best.pt",
    map_location="cpu",
    weights_only=True,
)

best_validation_loss = previous_best["validation"]["joint_loss"]
best_epoch = int(previous_best["epoch"])

RUN_DIR = NEW_RUN_DIR
RUN_DIR.mkdir()

shutil.copy2(PARENT_DIR / "best.pt", RUN_DIR / "best.pt")

with (PARENT_DIR / "run_config.json").open(
    encoding="utf-8"
) as config_file:
    provenance = json.load(config_file)

provenance["training"] = TRAIN_CONFIG
provenance["parent_checkpoint"] = str(PARENT_DIR / "last.pt")

with (RUN_DIR / "run_config.json").open(
    "w", encoding="utf-8"
) as config_file:
    json.dump(provenance, config_file, indent=2)

resumed_validation = evaluate(validation_data)

assert abs(
    resumed_validation["joint_loss"]
    - checkpoint["validation"]["joint_loss"]
) < 1e-5, "Checkpoint evaluation did not reproduce."

print(
    f"Resumed epoch {starting_epoch}; "
    f"validation loss={resumed_validation['joint_loss']:.6f}"
)

stop_reason = "max_additional_epochs"
additional_epochs = 0

torch.cuda.synchronize()
training_start = time.perf_counter()

for epoch in range(
    starting_epoch + 1,
    TRAIN_CONFIG["max_epochs"] + 1,
):
    if (
        time.perf_counter() - training_start
        >= TRAIN_CONFIG["training_time_limit_seconds"]
    ):
        stop_reason = "time_limit"
        break

    model.train()
    epoch_start = time.perf_counter()

    ordering = torch.randperm(
        len(train_data["tokens"]),
        generator=shuffle_generator,
    ).to(device)

    accumulated_loss = 0.0
    processed_tokens = 0
    processed_sequences = 0

    for start in range(
        0, len(ordering), TRAIN_CONFIG["batch_size"]
    ):
        if (
            time.perf_counter() - training_start
            >= TRAIN_CONFIG["training_time_limit_seconds"]
        ):
            stop_reason = "time_limit"
            break

        indices = ordering[
            start:start + TRAIN_CONFIG["batch_size"]
        ]

        optimizer.zero_grad(set_to_none=True)
        output = model(train_data["tokens"][indices])

        full_target = train_data["full_state"][indices, 1:]
        quotient_target = train_data["quotient_state"][indices, 1:]

        full_loss = F.cross_entropy(
            output["full_logits"][:, 1:].reshape(-1, 8),
            full_target.reshape(-1),
        )
        quotient_loss = F.cross_entropy(
            output["quotient_logits"][:, 1:].reshape(-1, 4),
            quotient_target.reshape(-1),
        )

        loss = 0.5 * (full_loss + quotient_loss)

        if not torch.isfinite(loss).item():
            raise RuntimeError(
                f"Nonfinite training loss at epoch {epoch}."
            )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=TRAIN_CONFIG["gradient_clip"],
            error_if_nonfinite=True,
        )
        optimizer.step()

        batch_tokens = full_target.numel()
        accumulated_loss += loss.item() * batch_tokens
        processed_tokens += batch_tokens
        processed_sequences += len(indices)

    if processed_tokens == 0:
        break

    validation = evaluate(validation_data)
    complete_epoch = processed_sequences == len(ordering)

    torch.cuda.synchronize()

    row = {
        "epoch": epoch,
        "complete_epoch": complete_epoch,
        "train_loss": accumulated_loss / processed_tokens,
        "epoch_seconds": time.perf_counter() - epoch_start,
        **validation,
    }

    history.append(row)
    additional_epochs += 1

    save_checkpoint(
        "last.pt", epoch, validation, complete_epoch
    )

    if validation["joint_loss"] < best_validation_loss:
        best_validation_loss = validation["joint_loss"]
        best_epoch = epoch
        save_checkpoint(
            "best.pt", epoch, validation, complete_epoch
        )

    with (RUN_DIR / "history.json").open(
        "w", encoding="utf-8"
    ) as history_file:
        json.dump(history, history_file, indent=2)

    if (
        epoch == starting_epoch + 1
        or epoch % 10 == 0
        or stop_reason == "time_limit"
    ):
        print(
            f"epoch={epoch:03d} | "
            f"train_loss={row['train_loss']:.4f} | "
            f"val_loss={validation['joint_loss']:.4f} | "
            f"full_token={validation['full_token_accuracy']:.2%} | "
            f"quotient_token={validation['quotient_token_accuracy']:.2%} | "
            f"full_sequence={validation['full_sequence_accuracy']:.2%} | "
            f"quotient_sequence={validation['quotient_sequence_accuracy']:.2%} | "
            f"complete={complete_epoch}",
            flush=True,
        )

    if stop_reason == "time_limit":
        break

selected_checkpoint = torch.load(
    RUN_DIR / "best.pt",
    map_location="cpu",
    weights_only=True,
)
model.load_state_dict(selected_checkpoint["model_state_dict"])

selected_train = evaluate(train_data)
selected_validation = evaluate(validation_data)

torch.cuda.synchronize()

summary = {
    "starting_epoch": starting_epoch,
    "additional_epochs_logged": additional_epochs,
    "best_epoch": best_epoch,
    "selected_train": selected_train,
    "selected_validation": selected_validation,
    "elapsed_seconds": time.perf_counter() - training_start,
    "stop_reason": stop_reason,
    "test_evaluated": False,
    "run_directory": str(RUN_DIR),
}

with (RUN_DIR / "summary.json").open(
    "w", encoding="utf-8"
) as summary_file:
    json.dump(summary, summary_file, indent=2)

print("\nCONTINUATION FINISHED")
print(json.dumps(summary, indent=2))

In [ ]:
import json
from pathlib import Path

import numpy as np
import torch

CHECKPOINT_DIR = Path(
    "q8_pilot_v1/training_continued_seed11"
)
DATA_DIR = Path("q8_pilot_v1")

device = torch.device(
    "cuda:0" if torch.cuda.is_available() else "cpu"
)

checkpoint = torch.load(
    CHECKPOINT_DIR / "best.pt",
    map_location="cpu",
    weights_only=True,
)

diagnostic_model = Q8Transformer(
    **checkpoint["model_config"]
).to(device)

diagnostic_model.load_state_dict(
    checkpoint["model_state_dict"]
)
diagnostic_model.eval()


@torch.no_grad()
def diagnose_split(split_name):
    with np.load(
        DATA_DIR / f"{split_name}.npz",
        allow_pickle=False,
    ) as archive:
        tokens = torch.from_numpy(
            archive["tokens"].copy()
        ).to(device)

        full_target = torch.from_numpy(
            archive["full_state"][:, 1:].copy()
        ).to(device)

        quotient_target = torch.from_numpy(
            archive["quotient_state"][:, 1:].copy()
        ).to(device)

        sign_target = torch.from_numpy(
            archive["nuisance"][:, 1:].copy()
        ).to(device)

        pair_ids = archive["pair_id"].copy()

    assert len(tokens) % 2 == 0
    assert np.array_equal(pair_ids[::2], pair_ids[1::2])
    assert np.array_equal(
        pair_ids[::2], np.arange(len(tokens) // 2)
    )

    assert torch.equal(
        tokens[::2, 1:], tokens[1::2, 1:]
    )
    assert torch.equal(
        quotient_target[::2], quotient_target[1::2]
    )
    assert torch.equal(
        sign_target[::2], 1 - sign_target[1::2]
    )

    full_batches = []
    quotient_batches = []

    for start in range(0, len(tokens), 256):
        output = diagnostic_model(tokens[start:start + 256])

        full_batches.append(output["full_logits"][:, 1:])
        quotient_batches.append(
            output["quotient_logits"][:, 1:]
        )

    full_logits = torch.cat(full_batches, dim=0)
    quotient_logits = torch.cat(quotient_batches, dim=0)

    assert torch.isfinite(full_logits).all()
    assert torch.isfinite(quotient_logits).all()

    full_prediction = full_logits.argmax(dim=-1)
    quotient_prediction = quotient_logits.argmax(dim=-1)

    quotient_from_full = full_prediction // 2

    state_probabilities = torch.softmax(full_logits, dim=-1)

    sign_probabilities = state_probabilities.reshape(
        len(tokens), full_target.shape[1], 4, 2
    ).sum(dim=2)

    marginal_sign_prediction = sign_probabilities.argmax(
        dim=-1
    )

    paired_state_logits = full_logits.reshape(
        len(tokens), full_target.shape[1], 4, 2
    )

    true_quotient_indices = (
        quotient_target.unsqueeze(-1)
        .unsqueeze(-1)
        .expand(-1, -1, 1, 2)
    )

    oracle_sign_logits = paired_state_logits.gather(
        dim=2,
        index=true_quotient_indices,
    ).squeeze(2)

    oracle_sign_prediction = oracle_sign_logits.argmax(
        dim=-1
    )

    full_matches = full_prediction == full_target
    quotient_matches = quotient_prediction == quotient_target

    quotient_from_full_matches = (
        quotient_from_full == quotient_target
    )
    marginal_sign_matches = (
        marginal_sign_prediction == sign_target
    )
    oracle_sign_matches = oracle_sign_prediction == sign_target

    quotient_pair_agreement = (
        quotient_prediction[::2]
        == quotient_prediction[1::2]
    )

    full_pair_equivariance = (
        (full_prediction[::2] ^ 1)
        == full_prediction[1::2]
    )

    def accuracy(matches):
        return matches.float().mean().item()

    summary = {
        "full_token_accuracy": accuracy(full_matches),
        "quotient_token_accuracy": accuracy(quotient_matches),
        "quotient_from_full_accuracy": accuracy(
            quotient_from_full_matches
        ),
        "marginal_sign_accuracy": accuracy(
            marginal_sign_matches
        ),
        "oracle_sign_accuracy": accuracy(oracle_sign_matches),
        "full_sequence_accuracy": accuracy(
            full_matches.all(dim=1)
        ),
        "quotient_sequence_accuracy": accuracy(
            quotient_matches.all(dim=1)
        ),
        "quotient_pair_agreement": accuracy(
            quotient_pair_agreement
        ),
        "full_pair_equivariance": accuracy(
            full_pair_equivariance
        ),
    }

    rows = []

    for position in range(full_target.shape[1]):
        rows.append({
            "position": position + 1,
            "full": accuracy(full_matches[:, position]),
            "quotient": accuracy(
                quotient_matches[:, position]
            ),
            "quotient_from_full": accuracy(
                quotient_from_full_matches[:, position]
            ),
            "sign_marginal": accuracy(
                marginal_sign_matches[:, position]
            ),
            "sign_oracle": accuracy(
                oracle_sign_matches[:, position]
            ),
        })

    print(f"\n--- {split_name.upper()} ---")
    print(json.dumps(summary, indent=2))

    print(
        "\nposition | full | quotient | "
        "quotient_from_full | sign_marginal | sign_oracle"
    )

    for row in rows:
        print(
            f"{row['position']:8d} | "
            f"{row['full']:6.2%} | "
            f"{row['quotient']:8.2%} | "
            f"{row['quotient_from_full']:18.2%} | "
            f"{row['sign_marginal']:13.2%} | "
            f"{row['sign_oracle']:11.2%}"
        )

    return {"summary": summary, "by_position": rows}


diagnostic_results = {
    "checkpoint_epoch": int(checkpoint["epoch"]),
    "test_evaluated": False,
    "train": diagnose_split("train"),
    "validation": diagnose_split("validation"),
}

assert abs(
    diagnostic_results["validation"]["summary"][
        "full_token_accuracy"
    ]
    - checkpoint["validation"]["full_token_accuracy"]
) < 1e-6

assert abs(
    diagnostic_results["validation"]["summary"][
        "quotient_token_accuracy"
    ]
    - checkpoint["validation"]["quotient_token_accuracy"]
) < 1e-6

with (CHECKPOINT_DIR / "position_diagnostics.json").open(
    "w", encoding="utf-8"
) as output_file:
    json.dump(diagnostic_results, output_file, indent=2)

print("\nPASS: checkpoint metrics reproduced.")
print("No training performed; test remains untouched.")

In [ ]:
import json
import shutil
import time
from pathlib import Path

import torch
import torch.nn.functional as F

assert torch.cuda.is_available(), "GPU is unavailable."
assert all(
    name in globals()
    for name in [
        "Q8Transformer",
        "evaluate",
        "save_checkpoint",
        "load_split",
        "shuffle_generator",
    ]
), "Use the same notebook session."

device = torch.device("cuda:0")

PARENT_DIR = Path("q8_pilot_v1/training_continued_seed11")
CURRICULUM_DIR = Path("q8_pilot_v1/curriculum_seed11")

assert not CURRICULUM_DIR.exists(), (
    "Curriculum results already exist; do not overwrite them."
)

checkpoint = torch.load(
    PARENT_DIR / "best.pt",
    map_location="cpu",
    weights_only=True,
)

MODEL_CONFIG = checkpoint["model_config"]
TRAIN_CONFIG = dict(checkpoint["train_config"])

TRAIN_CONFIG.update({
    "stage": "exploratory_prefix_curriculum",
    "parent_checkpoint": str(PARENT_DIR / "best.pt"),
    "curriculum_lengths": [4, 5, 6, 8, 10, 12],
    "max_epochs_per_length": 100,
    "training_time_limit_seconds": 300,
    "token_accuracy_gate": 0.995,
    "sequence_accuracy_gate": 0.95,
    "consecutive_gate_epochs": 3,
})

model = Q8Transformer(**MODEL_CONFIG).to(device)
model.load_state_dict(checkpoint["model_state_dict"])

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=TRAIN_CONFIG["learning_rate"],
    weight_decay=TRAIN_CONFIG["weight_decay"],
)
optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

shuffle_generator.set_state(
    checkpoint["shuffle_rng_state"].cpu()
)
torch.set_rng_state(checkpoint["torch_rng_state"].cpu())
torch.cuda.set_rng_state_all([
    state.cpu()
    for state in checkpoint["cuda_rng_states"]
])

train_data = load_split("train")
validation_data = load_split("validation")

RUN_DIR = CURRICULUM_DIR
RUN_DIR.mkdir()

with (PARENT_DIR / "run_config.json").open(
    encoding="utf-8"
) as config_file:
    provenance = json.load(config_file)

provenance["training"] = dict(TRAIN_CONFIG)

with (RUN_DIR / "run_config.json").open(
    "w", encoding="utf-8"
) as config_file:
    json.dump(provenance, config_file, indent=2)

global_epoch = int(checkpoint["epoch"])
history = []
stage_reports = []


def prefix_data(data, action_length):
    return {
        name: tensor[:, :action_length + 1]
        for name, tensor in data.items()
    }


def passes_gate(metrics):
    return (
        metrics["full_token_accuracy"]
        >= TRAIN_CONFIG["token_accuracy_gate"]
        and metrics["quotient_token_accuracy"]
        >= TRAIN_CONFIG["token_accuracy_gate"]
        and metrics["full_sequence_accuracy"]
        >= TRAIN_CONFIG["sequence_accuracy_gate"]
        and metrics["quotient_sequence_accuracy"]
        >= TRAIN_CONFIG["sequence_accuracy_gate"]
    )


def train_prefix_epoch(data, deadline):
    model.train()

    ordering = torch.randperm(
        len(data["tokens"]),
        generator=shuffle_generator,
    ).to(device)

    accumulated_loss = 0.0
    processed_tokens = 0
    processed_sequences = 0

    for start in range(
        0, len(ordering), TRAIN_CONFIG["batch_size"]
    ):
        if time.perf_counter() >= deadline:
            break

        indices = ordering[
            start:start + TRAIN_CONFIG["batch_size"]
        ]

        optimizer.zero_grad(set_to_none=True)
        output = model(data["tokens"][indices])

        full_target = data["full_state"][indices, 1:]
        quotient_target = data["quotient_state"][indices, 1:]

        full_loss = F.cross_entropy(
            output["full_logits"][:, 1:].reshape(-1, 8),
            full_target.reshape(-1),
        )
        quotient_loss = F.cross_entropy(
            output["quotient_logits"][:, 1:].reshape(-1, 4),
            quotient_target.reshape(-1),
        )

        loss = 0.5 * (full_loss + quotient_loss)

        if not torch.isfinite(loss).item():
            raise RuntimeError("Nonfinite loss during curriculum.")

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            TRAIN_CONFIG["gradient_clip"],
            error_if_nonfinite=True,
        )
        optimizer.step()

        token_count = full_target.numel()
        accumulated_loss += loss.item() * token_count
        processed_tokens += token_count
        processed_sequences += len(indices)

    return {
        "train_loss": (
            accumulated_loss / processed_tokens
            if processed_tokens else None
        ),
        "processed_sequences": processed_sequences,
        "complete_epoch": processed_sequences == len(ordering),
    }


print("Starting from checkpoint epoch:", global_epoch)
print("Initial length-12 validation:")
print(json.dumps(evaluate(validation_data), indent=2))

torch.cuda.synchronize()
curriculum_start = time.perf_counter()
deadline = (
    curriculum_start
    + TRAIN_CONFIG["training_time_limit_seconds"]
)

stop_reason = "all_stages_passed"

for action_length in TRAIN_CONFIG["curriculum_lengths"]:
    if time.perf_counter() >= deadline:
        stop_reason = "time_limit"
        break

    TRAIN_CONFIG["current_action_length"] = action_length

    stage_train = prefix_data(train_data, action_length)
    stage_validation = prefix_data(
        validation_data, action_length
    )

    initial_stage_metrics = evaluate(stage_validation)
    stage_best_loss = initial_stage_metrics["joint_loss"]

    best_filename = f"length_{action_length}_best.pt"
    gate_filename = f"length_{action_length}_passed.pt"

    save_checkpoint(
        best_filename,
        global_epoch,
        initial_stage_metrics,
        True,
    )

    consecutive_passes = 0
    stage_passed = False
    epochs_run = 0

    print(f"\n=== ACTION LENGTH {action_length} ===", flush=True)

    for stage_epoch in range(
        1, TRAIN_CONFIG["max_epochs_per_length"] + 1
    ):
        if time.perf_counter() >= deadline:
            stop_reason = "time_limit"
            break

        epoch_result = train_prefix_epoch(stage_train, deadline)

        if epoch_result["processed_sequences"] == 0:
            stop_reason = "time_limit"
            break

        global_epoch += 1
        epochs_run += 1

        metrics = evaluate(stage_validation)

        row = {
            "global_epoch": global_epoch,
            "stage_epoch": stage_epoch,
            "action_length": action_length,
            **epoch_result,
            **metrics,
        }
        history.append(row)

        save_checkpoint(
            "last.pt",
            global_epoch,
            metrics,
            epoch_result["complete_epoch"],
        )

        if metrics["joint_loss"] < stage_best_loss:
            stage_best_loss = metrics["joint_loss"]
            save_checkpoint(
                best_filename,
                global_epoch,
                metrics,
                epoch_result["complete_epoch"],
            )

        if epoch_result["complete_epoch"] and passes_gate(metrics):
            consecutive_passes += 1
        else:
            consecutive_passes = 0

        stage_passed = (
            consecutive_passes
            >= TRAIN_CONFIG["consecutive_gate_epochs"]
        )

        with (RUN_DIR / "history.json").open(
            "w", encoding="utf-8"
        ) as history_file:
            json.dump(history, history_file, indent=2)

        if stage_epoch == 1 or stage_epoch % 10 == 0 or stage_passed:
            print(
                f"length={action_length:02d} | "
                f"stage_epoch={stage_epoch:03d} | "
                f"full_token={metrics['full_token_accuracy']:.2%} | "
                f"quotient_token={metrics['quotient_token_accuracy']:.2%} | "
                f"full_sequence={metrics['full_sequence_accuracy']:.2%} | "
                f"quotient_sequence={metrics['quotient_sequence_accuracy']:.2%} | "
                f"gate_streak={consecutive_passes}",
                flush=True,
            )

        if stage_passed:
            save_checkpoint(
                gate_filename,
                global_epoch,
                metrics,
                True,
            )
            break

        if not epoch_result["complete_epoch"]:
            stop_reason = "time_limit"
            break

    selected_filename = (
        gate_filename if stage_passed else best_filename
    )

    selected = torch.load(
        RUN_DIR / selected_filename,
        map_location="cpu",
        weights_only=True,
    )
    model.load_state_dict(selected["model_state_dict"])

    current_length_metrics = evaluate(stage_validation)
    length12_metrics = evaluate(validation_data)

    stage_reports.append({
        "action_length": action_length,
        "passed": stage_passed,
        "epochs_run": epochs_run,
        "selected_checkpoint": selected_filename,
        "selected_global_epoch": int(selected["epoch"]),
        "current_length_validation": current_length_metrics,
        "length12_validation": length12_metrics,
    })

    print(
        f"Stage {'PASSED' if stage_passed else 'NOT PASSED'}; "
        f"length-12 full_token="
        f"{length12_metrics['full_token_accuracy']:.2%}; "
        f"length-12 quotient_token="
        f"{length12_metrics['quotient_token_accuracy']:.2%}",
        flush=True,
    )

    with (RUN_DIR / "stage_reports.json").open(
        "w", encoding="utf-8"
    ) as reports_file:
        json.dump(stage_reports, reports_file, indent=2)

    if not stage_passed:
        if stop_reason != "time_limit":
            stop_reason = f"length_{action_length}_gate_not_reached"
        break

assert stage_reports, (
    "No stage ran. Inspect the time limit and session state."
)

shutil.copy2(
    RUN_DIR / stage_reports[-1]["selected_checkpoint"],
    RUN_DIR / "selected.pt",
)

final_train = evaluate(train_data)
final_validation = evaluate(validation_data)

torch.cuda.synchronize()

summary = {
    "parent_epoch": int(checkpoint["epoch"]),
    "stop_reason": stop_reason,
    "stage_reports": stage_reports,
    "final_length12_train": final_train,
    "final_length12_validation": final_validation,
    "elapsed_seconds": time.perf_counter() - curriculum_start,
    "test_evaluated": False,
    "selected_checkpoint": str(RUN_DIR / "selected.pt"),
}

with (RUN_DIR / "summary.json").open(
    "w", encoding="utf-8"
) as summary_file:
    json.dump(summary, summary_file, indent=2)

print("\nCURRICULUM FINISHED")
print(json.dumps(summary, indent=2))

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import shutil
import zipfile
from IPython.display import display, FileLink

source_dir = Path("q8_pilot_v1")
assert source_dir.is_dir(), "Results folder not found."

timestamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
archive_name = f"q8_pilot_backup_{timestamp}"

archive_path = shutil.make_archive(
    archive_name,
    "zip",
    root_dir=".",
    base_dir=str(source_dir),
)

with zipfile.ZipFile(archive_path) as archive:
    assert archive.testzip() is None, "ZIP integrity check failed."
    print("Files in backup:", len(archive.namelist()))

print("Download this backup:", archive_path)
display(FileLink(archive_path))

In [ ]:
from pathlib import Path
import json
import zipfile

print("=== Kaggle runtime ===")
print("Python:", __import__("sys").version.split()[0])

try:
    import torch
    print("PyTorch:", torch.__version__)
    print("CUDA build:", torch.version.cuda)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
except Exception as exc:
    print("PyTorch check failed:", repr(exc))

print("\n=== Existing experiment folder ===")
root = Path("/kaggle/working/q8_pilot_v1")
print("Folder exists:", root.is_dir())

required = [
    "train.npz",
    "validation.npz",
    "test.npz",
    "training_continued_seed11/best.pt",
    "curriculum_seed11/summary.json",
    "curriculum_seed11/selected.pt",
]
if root.is_dir():
    for rel in required:
        path = root / rel
        size = f" ({path.stat().st_size:,} bytes)" if path.is_file() else ""
        print(f"{rel}: {path.is_file()}{size}")

    summary_path = root / "curriculum_seed11/summary.json"
    if summary_path.is_file():
        try:
            summary = json.loads(summary_path.read_text())
            print("Curriculum stop reason:", summary.get("stop_reason"))
            print("Curriculum test evaluated:", summary.get("test_evaluated"))
        except Exception as exc:
            print("Could not read curriculum summary:", repr(exc))

print("\n=== Backup ZIPs visible to this notebook ===")
search_dirs = [Path("/kaggle/working"), Path("/kaggle/input")]
archives = []
for folder in search_dirs:
    if folder.is_dir():
        archives.extend(folder.glob("*.zip"))
        archives.extend(folder.glob("*/*.zip"))

archives = list(dict.fromkeys(p for p in archives if p.is_file()))
if not archives:
    print("No ZIP backup found under /kaggle/working or /kaggle/input.")
else:
    for archive_path in archives:
        print(f"\n{archive_path} ({archive_path.stat().st_size:,} bytes)")
        try:
            with zipfile.ZipFile(archive_path) as archive:
                names = set(archive.namelist())
                for rel in required:
                    member = f"q8_pilot_v1/{rel}"
                    print(f"  {member}: {member in names}")
        except Exception as exc:
            print("  ZIP inspection failed:", repr(exc))

In [ ]:
from pathlib import Path
import json
import zipfile

archive_path = Path(
    "/kaggle/working/q8_pilot_backup_20261005_102000.zip"
)
working_summary = Path(
    "/kaggle/working/q8_pilot_v1/curriculum_seed11/summary.json"
)
member = "q8_pilot_v1/curriculum_seed11/summary.json"

print("Working summary bytes:",
      working_summary.stat().st_size if working_summary.exists() else "missing")

if not archive_path.is_file():
    print("Backup ZIP not found.")
else:
    with zipfile.ZipFile(archive_path) as archive:
        if member not in archive.namelist():
            print("Curriculum summary is missing from the ZIP.")
        else:
            info = archive.getinfo(member)
            raw = archive.read(member)
            print("ZIP summary bytes:", info.file_size)
            print("Bytes read:", len(raw))

            try:
                summary = json.loads(raw)
                print("Stop reason:", summary.get("stop_reason"))
                print("Test evaluated:", summary.get("test_evaluated"))
                print("Stages:", [
                    {
                        "length": stage.get("action_length"),
                        "passed": stage.get("passed"),
                        "epochs": stage.get("epochs_run"),
                    }
                    for stage in summary.get("stage_reports", [])
                ])
                print("Final length-12 validation:",
                      summary.get("final_length12_validation"))
            except Exception as exc:
                print("ZIP summary is not valid JSON:", repr(exc))

In [ ]:
from pathlib import Path
import json
import zipfile

archive_path = Path("/kaggle/working/q8_pilot_backup_20261005_102000.zip")
member = "q8_pilot_v1/curriculum_seed11/summary.json"
target = Path("/kaggle/working/q8_pilot_v1/curriculum_seed11/summary.json")

assert archive_path.is_file(), "Backup ZIP not found."
assert target.parent.is_dir(), "Curriculum folder not found."

with zipfile.ZipFile(archive_path) as archive:
    raw = archive.read(member)

summary = json.loads(raw)
assert summary.get("test_evaluated") is False

if target.exists() and target.stat().st_size != 0:
    raise RuntimeError(
        f"Refusing to overwrite non-empty file: {target}"
    )

target.write_bytes(raw)

print("Recovered summary bytes:", target.stat().st_size)
print("Stop reason:", summary.get("stop_reason"))
print("Test evaluated:", summary.get("test_evaluated"))
print("Final length-12 validation:",
      summary.get("final_length12_validation"))

print("\nNotebook objects currently available:")
for name in [
    "Q8Transformer",
    "evaluate",
    "save_checkpoint",
    "load_split",
    "shuffle_generator",
]:
    print(f"{name}: {name in globals()}")

In [ ]:
import json
import shutil
import time
from pathlib import Path

import torch
import torch.nn.functional as F

required_names = [
    "Q8Transformer",
    "evaluate",
    "save_checkpoint",
    "load_split",
    "shuffle_generator",
]

assert all(name in globals() for name in required_names), (
    "This cell requires the previous training session."
)
assert torch.cuda.is_available(), "GPU is unavailable."

device = torch.device("cuda:0")

PARENT_DIR = Path("q8_pilot_v1/training_continued_seed11")
NEW_RUN_DIR = Path("q8_pilot_v1/length12_control_seed11")

assert (PARENT_DIR / "best.pt").exists(), (
    "The parent best checkpoint is missing."
)
assert not NEW_RUN_DIR.exists(), (
    "Control results already exist. Do not overwrite them."
)

checkpoint = torch.load(
    PARENT_DIR / "best.pt",
    map_location="cpu",
    weights_only=True,
)

assert checkpoint["complete_epoch"], (
    "The checkpoint is from a partial epoch; resume needs extra handling."
)

MODEL_CONFIG = checkpoint["model_config"]
TRAIN_CONFIG = dict(checkpoint["train_config"])

starting_epoch = int(checkpoint["epoch"])

TRAIN_CONFIG.update({
    "max_epochs": starting_epoch + 119,
    "training_time_limit_seconds": 36.09,
    "parent_run": str(PARENT_DIR),
    "starting_epoch": starting_epoch,
    "stage": "matched_full_length12_continuation_control",
})

model = Q8Transformer(**MODEL_CONFIG).to(device)
model.load_state_dict(checkpoint["model_state_dict"])

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=TRAIN_CONFIG["learning_rate"],
    weight_decay=TRAIN_CONFIG["weight_decay"],
)
optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

shuffle_generator.set_state(
    checkpoint["shuffle_rng_state"].cpu()
)
torch.set_rng_state(checkpoint["torch_rng_state"].cpu())
torch.cuda.set_rng_state_all([
    state.cpu()
    for state in checkpoint["cuda_rng_states"]
])

train_data = load_split("train")
validation_data = load_split("validation")

with (PARENT_DIR / "history.json").open(
    encoding="utf-8"
) as history_file:
    history = json.load(history_file)

previous_best = torch.load(
    PARENT_DIR / "best.pt",
    map_location="cpu",
    weights_only=True,
)

best_validation_loss = previous_best["validation"]["joint_loss"]
best_epoch = int(previous_best["epoch"])

RUN_DIR = NEW_RUN_DIR
RUN_DIR.mkdir()

shutil.copy2(PARENT_DIR / "best.pt", RUN_DIR / "best.pt")

with (PARENT_DIR / "run_config.json").open(
    encoding="utf-8"
) as config_file:
    provenance = json.load(config_file)

provenance["training"] = TRAIN_CONFIG
provenance["parent_checkpoint"] = str(PARENT_DIR / "best.pt")

with (RUN_DIR / "run_config.json").open(
    "w", encoding="utf-8"
) as config_file:
    json.dump(provenance, config_file, indent=2)

resumed_validation = evaluate(validation_data)

assert abs(
    resumed_validation["joint_loss"]
    - checkpoint["validation"]["joint_loss"]
) < 1e-5, "Checkpoint evaluation did not reproduce."

print(
    f"Resumed epoch {starting_epoch}; "
    f"validation loss={resumed_validation['joint_loss']:.6f}"
)

stop_reason = "max_additional_epochs"
additional_epochs = 0

torch.cuda.synchronize()
training_start = time.perf_counter()

for epoch in range(
    starting_epoch + 1,
    TRAIN_CONFIG["max_epochs"] + 1,
):
    if (
        time.perf_counter() - training_start
        >= TRAIN_CONFIG["training_time_limit_seconds"]
    ):
        stop_reason = "time_limit"
        break

    model.train()
    epoch_start = time.perf_counter()

    ordering = torch.randperm(
        len(train_data["tokens"]),
        generator=shuffle_generator,
    ).to(device)

    accumulated_loss = 0.0
    processed_tokens = 0
    processed_sequences = 0

    for start in range(
        0, len(ordering), TRAIN_CONFIG["batch_size"]
    ):
        if (
            time.perf_counter() - training_start
            >= TRAIN_CONFIG["training_time_limit_seconds"]
        ):
            stop_reason = "time_limit"
            break

        indices = ordering[
            start:start + TRAIN_CONFIG["batch_size"]
        ]

        optimizer.zero_grad(set_to_none=True)
        output = model(train_data["tokens"][indices])

        full_target = train_data["full_state"][indices, 1:]
        quotient_target = train_data["quotient_state"][indices, 1:]

        full_loss = F.cross_entropy(
            output["full_logits"][:, 1:].reshape(-1, 8),
            full_target.reshape(-1),
        )
        quotient_loss = F.cross_entropy(
            output["quotient_logits"][:, 1:].reshape(-1, 4),
            quotient_target.reshape(-1),
        )

        loss = 0.5 * (full_loss + quotient_loss)

        if not torch.isfinite(loss).item():
            raise RuntimeError(
                f"Nonfinite training loss at epoch {epoch}."
            )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=TRAIN_CONFIG["gradient_clip"],
            error_if_nonfinite=True,
        )
        optimizer.step()

        batch_tokens = full_target.numel()
        accumulated_loss += loss.item() * batch_tokens
        processed_tokens += batch_tokens
        processed_sequences += len(indices)

    if processed_tokens == 0:
        break

    validation = evaluate(validation_data)
    complete_epoch = processed_sequences == len(ordering)

    torch.cuda.synchronize()

    row = {
        "epoch": epoch,
        "complete_epoch": complete_epoch,
        "train_loss": accumulated_loss / processed_tokens,
        "epoch_seconds": time.perf_counter() - epoch_start,
        **validation,
    }

    history.append(row)
    additional_epochs += 1

    save_checkpoint(
        "last.pt", epoch, validation, complete_epoch
    )

    if validation["joint_loss"] < best_validation_loss:
        best_validation_loss = validation["joint_loss"]
        best_epoch = epoch
        save_checkpoint(
            "best.pt", epoch, validation, complete_epoch
        )

    with (RUN_DIR / "history.json").open(
        "w", encoding="utf-8"
    ) as history_file:
        json.dump(history, history_file, indent=2)

    if (
        epoch == starting_epoch + 1
        or epoch % 10 == 0
        or stop_reason == "time_limit"
    ):
        print(
            f"epoch={epoch:03d} | "
            f"train_loss={row['train_loss']:.4f} | "
            f"val_loss={validation['joint_loss']:.4f} | "
            f"full_token={validation['full_token_accuracy']:.2%} | "
            f"quotient_token={validation['quotient_token_accuracy']:.2%} | "
            f"full_sequence={validation['full_sequence_accuracy']:.2%} | "
            f"quotient_sequence={validation['quotient_sequence_accuracy']:.2%} | "
            f"complete={complete_epoch}",
            flush=True,
        )

    if stop_reason == "time_limit":
        break

selected_checkpoint = torch.load(
    RUN_DIR / "best.pt",
    map_location="cpu",
    weights_only=True,
)
model.load_state_dict(selected_checkpoint["model_state_dict"])

selected_train = evaluate(train_data)
selected_validation = evaluate(validation_data)

torch.cuda.synchronize()

summary = {
    "starting_epoch": starting_epoch,
    "additional_epochs_logged": additional_epochs,
    "best_epoch": best_epoch,
    "selected_train": selected_train,
    "selected_validation": selected_validation,
    "elapsed_seconds": time.perf_counter() - training_start,
    "stop_reason": stop_reason,
    "test_evaluated": False,
    "run_directory": str(RUN_DIR),
}

with (RUN_DIR / "summary.json").open(
    "w", encoding="utf-8"
) as summary_file:
    json.dump(summary, summary_file, indent=2)

print("\nCONTINUATION FINISHED")
print(json.dumps(summary, indent=2))

In [ ]:
# Q8: compare the existing dual head with a quotient-conditioned sign head.
# Run this cell by itself. It does not load or evaluate test.npz.

import hashlib
import json
import math
import random
import time
import zipfile
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from torch import nn


# ---------- Runtime and input checks ----------

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU در Kaggle فعال نیست. از Notebook options بخش Accelerator را روی GPU بگذار."
    )

device = torch.device("cuda:0")
print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))

EXPECTED_ARCHIVE_SHA256 = (
    "bb1678db82f93282acd69d426fb46c348dd9ee5ec902ed8b77f87b2c3d2ada50"
)
EXPECTED_DATA_SHA256 = {
    "train": "8ff1da09a7935b9db3dd0ba3e212c258dd024a9fb21112b25ec95730e283c1a4",
    "validation": "1ab60070506a3941f385d72c2ca36760e3bbe5998aecae3a8a766c86d74c9a8b",
}

archive_candidates = []
for search_root in (
    Path.cwd(),
    Path("/kaggle/working"),
    Path("/kaggle/input"),
):
    if search_root.exists():
        archive_candidates.extend(
            search_root.rglob("q8_pilot_backup_20261005_102000.zip")
        )

archive_candidates = sorted(set(archive_candidates))
if not archive_candidates:
    raise FileNotFoundError(
        "فایل q8_pilot_backup_20261005_102000.zip پیدا نشد. "
        "آن را در Kaggle به‌عنوان Input اضافه کن."
    )

archive_path = archive_candidates[0]
archive_sha256 = hashlib.sha256(archive_path.read_bytes()).hexdigest()
if archive_sha256 != EXPECTED_ARCHIVE_SHA256:
    raise RuntimeError(
        f"هش بکاپ با نسخه‌ی تأییدشده فرق دارد: {archive_sha256}"
    )

data_dir = Path.cwd() / "q8_head_comparison_data"
data_dir.mkdir(exist_ok=True)

# فقط همین سه عضو خوانده می‌شوند. test.npz باز یا استخراج نمی‌شود.
with zipfile.ZipFile(archive_path, "r") as archive:
    for member in (
        "q8_pilot_v1/train.npz",
        "q8_pilot_v1/validation.npz",
        "q8_pilot_v1/data_config.json",
    ):
        destination = data_dir / Path(member).name
        destination.write_bytes(archive.read(member))

for split_name, expected_hash in EXPECTED_DATA_SHA256.items():
    actual_hash = hashlib.sha256(
        (data_dir / f"{split_name}.npz").read_bytes()
    ).hexdigest()
    if actual_hash != expected_hash:
        raise RuntimeError(f"هش فایل {split_name} تأیید نشد: {actual_hash}")

with (data_dir / "data_config.json").open(encoding="utf-8") as file:
    data_config = json.load(file)

print("بکاپ و هش‌های train/validation تأیید شدند؛ test خوانده نشده است.")


# ---------- Fixed model and data configuration ----------

MODEL_CONFIG = {
    "vocab_size": 12,
    "d_model": 64,
    "n_heads": 4,
    "n_layers": 3,
    "d_ff": 128,
    "max_tokens": 65,
    "dropout": 0.0,
}

TRAIN_CONFIG = {
    "model_seed": 11,
    "shuffle_seed": 1011,
    "batch_size": 256,
    "learning_rate": 0.001,
    "weight_decay": 0.01,
    "epochs": 200,
    "gradient_clip": 1.0,
    "loss": "0.5 * (full-state NLL + quotient cross-entropy)",
}

def load_split(split_name):
    with np.load(data_dir / f"{split_name}.npz", allow_pickle=False) as data:
        result = {
            name: torch.from_numpy(data[name].copy()).to(device)
            for name in ("tokens", "full_state", "quotient_state")
        }
    return result

train_data = load_split("train")
validation_data = load_split("validation")

assert train_data["tokens"].shape == (8192, 13)
assert validation_data["tokens"].shape == (1024, 13)
assert len(train_data["tokens"]) % TRAIN_CONFIG["batch_size"] == 0

optimizer_updates_per_epoch = (
    len(train_data["tokens"]) // TRAIN_CONFIG["batch_size"]
)
assert optimizer_updates_per_epoch == 32
assert optimizer_updates_per_epoch * TRAIN_CONFIG["epochs"] == 6400


# ---------- Same Transformer backbone, two output-head choices ----------

class CausalBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout):
        super().__init__()
        self.attention_norm = nn.LayerNorm(d_model)
        self.attention = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=n_heads,
            dropout=dropout,
            batch_first=True,
        )
        self.feedforward_norm = nn.LayerNorm(d_model)
        self.feedforward = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
        )

    def forward(self, hidden, attention_mask):
        normalized = self.attention_norm(hidden)
        attended, _ = self.attention(
            normalized,
            normalized,
            normalized,
            attn_mask=attention_mask,
            need_weights=False,
        )
        hidden = hidden + attended
        hidden = hidden + self.feedforward(
            self.feedforward_norm(hidden)
        )
        return hidden


class HeadComparisonQ8Transformer(nn.Module):
    def __init__(self, head_type):
        super().__init__()
        if head_type not in ("dual", "conditional"):
            raise ValueError(f"Unknown head type: {head_type}")

        d_model = MODEL_CONFIG["d_model"]
        self.head_type = head_type
        self.token_embedding = nn.Embedding(
            MODEL_CONFIG["vocab_size"], d_model
        )

        positions = torch.arange(
            MODEL_CONFIG["max_tokens"], dtype=torch.float32
        ).unsqueeze(1)
        frequencies = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float32)
            * (-math.log(10000.0) / d_model)
        )
        positional_encoding = torch.zeros(
            MODEL_CONFIG["max_tokens"], d_model
        )
        positional_encoding[:, 0::2] = torch.sin(positions * frequencies)
        positional_encoding[:, 1::2] = torch.cos(positions * frequencies)
        self.register_buffer("positional_encoding", positional_encoding)

        self.blocks = nn.ModuleList([
            CausalBlock(
                d_model,
                MODEL_CONFIG["n_heads"],
                MODEL_CONFIG["d_ff"],
                MODEL_CONFIG["dropout"],
            )
            for _ in range(MODEL_CONFIG["n_layers"])
        ])
        self.final_norm = nn.LayerNorm(d_model)

        # Both designs have 12 output values per hidden unit.
        self.quotient_head = nn.Linear(d_model, 4)
        if head_type == "dual":
            self.full_head = nn.Linear(d_model, 8)
        else:
            self.conditional_sign_head = nn.Linear(d_model, 8)

    def forward(self, tokens):
        sequence_length = tokens.shape[1]
        if sequence_length > self.positional_encoding.shape[0]:
            raise ValueError("Sequence is longer than configured maximum.")

        hidden = (
            self.token_embedding(tokens)
            + self.positional_encoding[:sequence_length].unsqueeze(0)
        )

        attention_mask = torch.triu(
            torch.ones(
                sequence_length,
                sequence_length,
                dtype=torch.bool,
                device=tokens.device,
            ),
            diagonal=1,
        )

        for block in self.blocks:
            hidden = block(hidden, attention_mask)

        hidden = self.final_norm(hidden)
        quotient_logits = self.quotient_head(hidden)

        if self.head_type == "dual":
            full_scores = self.full_head(hidden)
            sign_logits = None
        else:
            # P(full state) = P(quotient) * P(sign | quotient)
            sign_logits = self.conditional_sign_head(hidden).reshape(
                *hidden.shape[:-1], 4, 2
            )
            full_scores = (
                F.log_softmax(quotient_logits, dim=-1).unsqueeze(-1)
                + F.log_softmax(sign_logits, dim=-1)
            ).flatten(start_dim=-2)

        return {
            "full_scores": full_scores,
            "quotient_logits": quotient_logits,
            "sign_logits": sign_logits,
        }


def output_head_parameter_count(model):
    prefixes = (
        ("quotient_head.", "full_head.")
        if model.head_type == "dual"
        else ("quotient_head.", "conditional_sign_head.")
    )
    return sum(
        parameter.numel()
        for name, parameter in model.named_parameters()
        if name.startswith(prefixes)
    )


# Check equal head parameter counts and normalized conditional probabilities.
torch.manual_seed(TRAIN_CONFIG["model_seed"])
dual_probe = HeadComparisonQ8Transformer("dual").to(device)
conditional_probe = HeadComparisonQ8Transformer("conditional").to(device)

assert output_head_parameter_count(dual_probe) == 780
assert output_head_parameter_count(conditional_probe) == 780
assert sum(p.numel() for p in dual_probe.parameters()) == 102092
assert sum(p.numel() for p in conditional_probe.parameters()) == 102092

with torch.no_grad():
    probe = conditional_probe(validation_data["tokens"][:2])
    max_log_normalization_error = (
        torch.logsumexp(probe["full_scores"], dim=-1).abs().max().item()
    )
assert max_log_normalization_error < 2e-6
del dual_probe, conditional_probe, probe
torch.cuda.empty_cache()

print("Head checks passed: 780 output parameters per design.")


# ---------- Metrics and training ----------

def evaluate(model, head_type, data):
    model.eval()
    full_loss_sum = 0.0
    quotient_loss_sum = 0.0
    full_correct = 0
    quotient_correct = 0
    full_sequence_correct = 0
    quotient_sequence_correct = 0
    token_count = 0
    sequence_count = len(data["tokens"])
    position_correct = torch.zeros(12, dtype=torch.int64, device=device)
    sign_true_q_correct = 0
    sign_pred_q_correct = 0

    full_loss_fn = (
        F.nll_loss if head_type == "conditional" else F.cross_entropy
    )

    with torch.no_grad():
        for start in range(0, sequence_count, TRAIN_CONFIG["batch_size"]):
            stop = start + TRAIN_CONFIG["batch_size"]
            output = model(data["tokens"][start:stop])

            full_scores = output["full_scores"][:, 1:]
            quotient_logits = output["quotient_logits"][:, 1:]
            full_target = data["full_state"][start:stop, 1:]
            quotient_target = data["quotient_state"][start:stop, 1:]

            full_loss_sum += full_loss_fn(
                full_scores.reshape(-1, 8),
                full_target.reshape(-1),
                reduction="sum",
            ).item()
            quotient_loss_sum += F.cross_entropy(
                quotient_logits.reshape(-1, 4),
                quotient_target.reshape(-1),
                reduction="sum",
            ).item()

            full_matches = full_scores.argmax(-1) == full_target
            quotient_matches = quotient_logits.argmax(-1) == quotient_target

            full_correct += full_matches.sum().item()
            quotient_correct += quotient_matches.sum().item()
            full_sequence_correct += full_matches.all(dim=1).sum().item()
            quotient_sequence_correct += quotient_matches.all(dim=1).sum().item()
            position_correct += full_matches.sum(dim=0)
            token_count += full_target.numel()

            if head_type == "conditional":
                sign_logits = output["sign_logits"][:, 1:]
                predicted_q = quotient_logits.argmax(-1)

                true_q_sign = sign_logits.gather(
                    2,
                    quotient_target[..., None, None].expand(-1, -1, 1, 2),
                ).squeeze(2)
                predicted_q_sign = sign_logits.gather(
                    2,
                    predicted_q[..., None, None].expand(-1, -1, 1, 2),
                ).squeeze(2)

                sign_target = full_target % 2
                sign_true_q_correct += (
                    true_q_sign.argmax(-1) == sign_target
                ).sum().item()
                sign_pred_q_correct += (
                    predicted_q_sign.argmax(-1) == sign_target
                ).sum().item()

    full_loss = full_loss_sum / token_count
    quotient_loss = quotient_loss_sum / token_count

    return {
        "joint_loss": 0.5 * (full_loss + quotient_loss),
        "full_loss": full_loss,
        "quotient_loss": quotient_loss,
        "full_token_accuracy": full_correct / token_count,
        "quotient_token_accuracy": quotient_correct / token_count,
        "full_sequence_accuracy": full_sequence_correct / sequence_count,
        "quotient_sequence_accuracy": quotient_sequence_correct / sequence_count,
        "full_token_accuracy_by_position": (
            position_correct.float().div(sequence_count).cpu().tolist()
        ),
        "sign_accuracy_given_true_quotient": (
            sign_true_q_correct / token_count
            if head_type == "conditional" else None
        ),
        "sign_accuracy_given_predicted_quotient": (
            sign_pred_q_correct / token_count
            if head_type == "conditional" else None
        ),
    }


output_root = Path.cwd() / "q8_head_comparison_seed11"
if output_root.exists():
    raise FileExistsError(
        f"{output_root} already exists. Do not overwrite it; "
        "use a fresh Kaggle session or rename the output directory."
    )
output_root.mkdir()

data_hashes = {
    split: hashlib.sha256(
        (data_dir / f"{split}.npz").read_bytes()
    ).hexdigest()
    for split in ("train", "validation")
}

all_summaries = {}

for head_type in ("dual", "conditional"):
    run_dir = output_root / head_type
    run_dir.mkdir()

    # Reset seeds so both arms use the same initialization/data ordering seed.
    random.seed(TRAIN_CONFIG["model_seed"])
    np.random.seed(TRAIN_CONFIG["model_seed"])
    torch.manual_seed(TRAIN_CONFIG["model_seed"])
    torch.cuda.manual_seed_all(TRAIN_CONFIG["model_seed"])

    model = HeadComparisonQ8Transformer(head_type).to(device)
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=TRAIN_CONFIG["learning_rate"],
        weight_decay=TRAIN_CONFIG["weight_decay"],
    )
    shuffle_generator = torch.Generator()
    shuffle_generator.manual_seed(TRAIN_CONFIG["shuffle_seed"])

    run_config = {
        "head_type": head_type,
        "model_config": MODEL_CONFIG,
        "train_config": TRAIN_CONFIG,
        "source_archive_sha256": archive_sha256,
        "data_sha256": data_hashes,
        "gpu": torch.cuda.get_device_name(0),
        "torch_version": str(torch.__version__),
        "data_config": data_config,
        "output_head_parameter_count": output_head_parameter_count(model),
        "optimizer_updates_per_epoch": optimizer_updates_per_epoch,
        "optimizer_updates_target": 6400,
        "test_evaluated": False,
    }
    (run_dir / "run_config.json").write_text(
        json.dumps(run_config, indent=2), encoding="utf-8"
    )

    initial_validation = evaluate(model, head_type, validation_data)
    history = []
    best_validation_loss = float("inf")
    best_epoch = None
    gate_streak = 0
    gate_passed_epoch = None

    def save_checkpoint(filename, epoch, validation):
        checkpoint = {
            "model_state_dict": {
                name: tensor.detach().cpu()
                for name, tensor in model.state_dict().items()
            },
            "optimizer_state_dict": optimizer.state_dict(),
            "epoch": epoch,
            "validation": validation,
            "head_type": head_type,
            "model_config": MODEL_CONFIG,
            "train_config": TRAIN_CONFIG,
            "shuffle_rng_state": shuffle_generator.get_state(),
            "torch_rng_state": torch.get_rng_state(),
            "cuda_rng_states": torch.cuda.get_rng_state_all(),
        }
        temporary_path = run_dir / f"{filename}.tmp"
        torch.save(checkpoint, temporary_path)
        temporary_path.replace(run_dir / filename)

    start_time = time.perf_counter()
    print(f"\nStarting {head_type} head: 200 epochs, 6,400 updates.")

    for epoch in range(1, TRAIN_CONFIG["epochs"] + 1):
        model.train()
        epoch_start = time.perf_counter()
        ordering = torch.randperm(
            len(train_data["tokens"]),
            generator=shuffle_generator,
        ).to(device)

        accumulated_loss = 0.0
        processed_tokens = 0

        for start in range(
            0, len(ordering), TRAIN_CONFIG["batch_size"]
        ):
            indices = ordering[start:start + TRAIN_CONFIG["batch_size"]]
            output = model(train_data["tokens"][indices])

            full_target = train_data["full_state"][indices, 1:]
            quotient_target = train_data["quotient_state"][indices, 1:]

            full_loss_fn = (
                F.nll_loss if head_type == "conditional"
                else F.cross_entropy
            )
            full_loss = full_loss_fn(
                output["full_scores"][:, 1:].reshape(-1, 8),
                full_target.reshape(-1),
            )
            quotient_loss = F.cross_entropy(
                output["quotient_logits"][:, 1:].reshape(-1, 4),
                quotient_target.reshape(-1),
            )
            loss = 0.5 * (full_loss + quotient_loss)

            if not torch.isfinite(loss).item():
                raise RuntimeError(f"Nonfinite loss in epoch {epoch}.")

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=TRAIN_CONFIG["gradient_clip"],
                error_if_nonfinite=True,
            )
            optimizer.step()

            batch_tokens = full_target.numel()
            accumulated_loss += loss.item() * batch_tokens
            processed_tokens += batch_tokens

        if processed_tokens != 8192 * 12:
            raise RuntimeError(
                f"Epoch {epoch} was incomplete: {processed_tokens} target tokens."
            )

        validation = evaluate(model, head_type, validation_data)
        row = {
            "epoch": epoch,
            "complete_epoch": True,
            "train_loss": accumulated_loss / processed_tokens,
            "epoch_seconds": time.perf_counter() - epoch_start,
            **validation,
        }
        history.append(row)

        gate_ok = (
            validation["full_token_accuracy"] >= 0.995
            and validation["quotient_token_accuracy"] >= 0.995
            and validation["full_sequence_accuracy"] >= 0.95
            and validation["quotient_sequence_accuracy"] >= 0.95
        )
        gate_streak = gate_streak + 1 if gate_ok else 0
        if gate_streak >= 3 and gate_passed_epoch is None:
            gate_passed_epoch = epoch - 2

        save_checkpoint("last.pt", epoch, validation)
        if validation["joint_loss"] < best_validation_loss:
            best_validation_loss = validation["joint_loss"]
            best_epoch = epoch
            save_checkpoint("best.pt", epoch, validation)

        (run_dir / "history.json").write_text(
            json.dumps(history, indent=2), encoding="utf-8"
        )

        if epoch == 1 or epoch % 10 == 0 or epoch == TRAIN_CONFIG["epochs"]:
            print(
                f"{head_type} epoch={epoch:03d} "
                f"val full-token={validation['full_token_accuracy']:.2%} "
                f"quotient-token={validation['quotient_token_accuracy']:.2%} "
                f"full-seq={validation['full_sequence_accuracy']:.2%} "
                f"quotient-seq={validation['quotient_sequence_accuracy']:.2%}"
            )

    best_validation = next(
        row for row in history if row["epoch"] == best_epoch
    )
    summary = {
        "head_type": head_type,
        "initial_validation": initial_validation,
        "best_validation": best_validation,
        "last_validation": history[-1],
        "best_epoch": best_epoch,
        "best_validation_loss": best_validation_loss,
        "epochs_logged": len(history),
        "optimizer_updates": len(history) * optimizer_updates_per_epoch,
        "elapsed_seconds": time.perf_counter() - start_time,
        "gate_passed_epoch": gate_passed_epoch,
        "success_gate": (
            "3 consecutive validation epochs: both token accuracies >=99.5% "
            "and both exact-sequence accuracies >=95%"
        ),
        "test_evaluated": False,
    }
    (run_dir / "summary.json").write_text(
        json.dumps(summary, indent=2), encoding="utf-8"
    )
    all_summaries[head_type] = summary
    del model, optimizer
    torch.cuda.empty_cache()


print("\n=== Validation comparison ===")
for head_type, summary in all_summaries.items():
    best = summary["best_validation"]
    print(
        f"{head_type}: best epoch {summary['best_epoch']}; "
        f"full token {best['full_token_accuracy']:.3%}; "
        f"quotient token {best['quotient_token_accuracy']:.3%}; "
        f"full sequence {best['full_sequence_accuracy']:.3%}; "
        f"quotient sequence {best['quotient_sequence_accuracy']:.3%}; "
        f"gate epoch {summary['gate_passed_epoch']}"
    )

result_zip = Path.cwd() / "q8_head_comparison_results_seed11.zip"
with zipfile.ZipFile(
    result_zip, "w", compression=zipfile.ZIP_DEFLATED
) as archive:
    for result_file in sorted(output_root.rglob("*")):
        if result_file.is_file():
            archive.write(result_file, result_file.relative_to(output_root))

print("\nتمام شد. این فایل را از Kaggle دانلود و همین‌جا بارگذاری کن:")
print(result_zip)
print("test_evaluated = False")

In [ ]:
# Q8 sign-information probe
# Reads only train.npz and validation.npz; never opens the test split.
# Probes the final hidden representation (immediately before the output heads).

import hashlib
import io
import json
import shutil
import zipfile
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import torch
from torch import nn

from sklearn.metrics import accuracy_score
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression


# ---------- Locate the two input archives ----------

RESULTS_ZIP_NAME = "q8_head_comparison_results_seed11.zip"
DATA_ZIP_NAME = "q8_pilot_backup_20261005_102000.zip"

def find_input_file(filename):
    roots = [Path("/kaggle/input"), Path("/kaggle/working"), Path.cwd()]
    seen = set()

    for root in roots:
        try:
            resolved = root.resolve()
        except Exception:
            resolved = root

        if resolved in seen or not root.exists():
            continue
        seen.add(resolved)

        matches = [
            path for path in root.rglob(filename)
            if path.is_file()
        ]
        if matches:
            return matches[0]

    raise FileNotFoundError(
        f"Cannot find {filename}. Add it to this Kaggle notebook's Inputs "
        "and run this cell again."
    )


results_zip_path = find_input_file(RESULTS_ZIP_NAME)
print("Results archive:", results_zip_path)

# The results archive contains the exact hashes expected for train/validation.
with zipfile.ZipFile(results_zip_path, "r") as results_zip:
    run_config = json.loads(
        results_zip.read("conditional/run_config.json")
    )
    conditional_summary = json.loads(
        results_zip.read("conditional/summary.json")
    )
    checkpoint_bytes = results_zip.read("conditional/best.pt")

assert run_config["head_type"] == "conditional", (
    "The supplied archive is not the conditional-head run."
)

expected_hashes = run_config["data_sha256"]


# ---------- Load only the matching train/validation data ----------

def sha256_bytes(data_bytes):
    return hashlib.sha256(data_bytes).hexdigest()


def load_split_bytes(split_name):
    expected_hash = expected_hashes[split_name]
    member_name = f"q8_pilot_v1/{split_name}.npz"

    # Reuse local files only if their hash matches the conditional run config.
    local_candidates = [
        Path("q8_pilot_v1") / f"{split_name}.npz",
        Path("/kaggle/working/q8_pilot_v1") / f"{split_name}.npz",
    ]

    for path in local_candidates:
        if path.is_file():
            candidate_bytes = path.read_bytes()
            if sha256_bytes(candidate_bytes) == expected_hash:
                print(f"{split_name}: using matching local file")
                return candidate_bytes

    # Otherwise read only the requested train/validation member from the backup.
    data_zip_path = find_input_file(DATA_ZIP_NAME)
    with zipfile.ZipFile(data_zip_path, "r") as data_zip:
        candidate_bytes = data_zip.read(member_name)

    actual_hash = sha256_bytes(candidate_bytes)
    assert actual_hash == expected_hash, (
        f"{split_name} hash mismatch: expected {expected_hash}, "
        f"got {actual_hash}"
    )
    print(f"{split_name}: verified hash from backup archive")
    return candidate_bytes


def load_split(split_name):
    split_bytes = load_split_bytes(split_name)
    with np.load(io.BytesIO(split_bytes), allow_pickle=False) as stored:
        required = [
            "tokens",
            "full_state",
            "quotient_state",
            "nuisance",
            "pair_id",
            "sequence_id",
        ]
        result = {key: stored[key].copy() for key in required}

    assert result["tokens"].shape == result["full_state"].shape
    assert result["full_state"].shape == result["quotient_state"].shape
    assert result["full_state"].shape == result["nuisance"].shape
    return result


train_data = load_split("train")
validation_data = load_split("validation")

assert train_data["tokens"].shape[1] == 13
assert validation_data["tokens"].shape[1] == 13
assert np.array_equal(train_data["nuisance"], train_data["full_state"] % 2)
assert np.array_equal(
    validation_data["nuisance"],
    validation_data["full_state"] % 2,
)

print("Train rows:", len(train_data["tokens"]))
print("Validation rows:", len(validation_data["tokens"]))
print("Test split: not opened or evaluated")


# ---------- Rebuild the checkpoint-compatible Q8 Transformer ----------

class ProbeCausalBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout):
        super().__init__()
        self.attention_norm = nn.LayerNorm(d_model)
        self.attention = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=n_heads,
            dropout=dropout,
            batch_first=True,
        )
        self.feedforward_norm = nn.LayerNorm(d_model)
        self.feedforward = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
        )

    def forward(self, hidden, attention_mask):
        normalized = self.attention_norm(hidden)
        attended, _ = self.attention(
            normalized,
            normalized,
            normalized,
            attn_mask=attention_mask,
            need_weights=False,
        )
        hidden = hidden + attended
        hidden = hidden + self.feedforward(
            self.feedforward_norm(hidden)
        )
        return hidden


class Q8ProbeBackbone(nn.Module):
    def __init__(
        self,
        vocab_size,
        d_model,
        n_heads,
        n_layers,
        d_ff,
        max_tokens,
        dropout,
    ):
        super().__init__()

        self.token_embedding = nn.Embedding(vocab_size, d_model)

        positions = torch.arange(
            max_tokens, dtype=torch.float32
        ).unsqueeze(1)
        frequencies = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float32)
            * (-np.log(10000.0) / d_model)
        )

        positional_encoding = torch.zeros(max_tokens, d_model)
        positional_encoding[:, 0::2] = torch.sin(
            positions * frequencies
        )
        positional_encoding[:, 1::2] = torch.cos(
            positions * frequencies
        )
        self.register_buffer("positional_encoding", positional_encoding)

        self.blocks = nn.ModuleList([
            ProbeCausalBlock(d_model, n_heads, d_ff, dropout)
            for _ in range(n_layers)
        ])

        self.final_norm = nn.LayerNorm(d_model)

        # This head is only a placeholder so the conditional checkpoint can
        # load strictly after its 8 sign logits are mapped to this 8-wide layer.
        self.full_head = nn.Linear(d_model, 8)
        self.quotient_head = nn.Linear(d_model, 4)

    def forward(self, tokens):
        sequence_length = tokens.shape[1]
        if sequence_length > self.positional_encoding.shape[0]:
            raise ValueError("Sequence is longer than the configured maximum.")

        hidden = (
            self.token_embedding(tokens)
            + self.positional_encoding[:sequence_length].unsqueeze(0)
        )

        attention_mask = torch.triu(
            torch.ones(
                sequence_length,
                sequence_length,
                dtype=torch.bool,
                device=tokens.device,
            ),
            diagonal=1,
        )

        for block in self.blocks:
            hidden = block(hidden, attention_mask)

        return self.final_norm(hidden)


checkpoint = torch.load(
    io.BytesIO(checkpoint_bytes),
    map_location="cpu",
    weights_only=False,
)

model_config = checkpoint["model_config"]
probe_model = Q8ProbeBackbone(**model_config)

state_dict = dict(checkpoint["model_state_dict"])
assert "conditional_sign_head.weight" in state_dict
assert "conditional_sign_head.bias" in state_dict

# The conditional sign head has 8 logits (4 quotients x 2 signs).
# Its weights are not used for probing, but this mapping lets us load the
# shared Transformer backbone with strict key/shape checking.
state_dict["full_head.weight"] = state_dict.pop(
    "conditional_sign_head.weight"
)
state_dict["full_head.bias"] = state_dict.pop(
    "conditional_sign_head.bias"
)

probe_model.load_state_dict(state_dict, strict=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
probe_model = probe_model.to(device).eval()
print("Activation extraction device:", device)
print("Checkpoint epoch:", checkpoint.get("epoch"))


# ---------- Extract final hidden states for train and validation ----------

@torch.inference_mode()
def extract_final_hidden(data, batch_size=256):
    all_hidden = []

    tokens = data["tokens"]
    for start in range(0, len(tokens), batch_size):
        token_batch = torch.as_tensor(
            tokens[start:start + batch_size],
            dtype=torch.long,
            device=device,
        )
        hidden = probe_model(token_batch)
        all_hidden.append(hidden.cpu().numpy().astype(np.float32))

    return np.concatenate(all_hidden, axis=0)


train_h = extract_final_hidden(train_data)
validation_h = extract_final_hidden(validation_data)

assert train_h.shape[:2] == train_data["full_state"].shape
assert validation_h.shape[:2] == validation_data["full_state"].shape
print("Hidden-state shape, train:", train_h.shape)
print("Hidden-state shape, validation:", validation_h.shape)


# ---------- Save compressed activation exports for this diagnostic ----------

run_stamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
run_dir = Path(f"q8_sign_probe_seed11_{run_stamp}")
run_dir.mkdir(parents=True, exist_ok=False)


def save_activation_export(split_name, data, hidden):
    n_sequences, n_positions, _ = hidden.shape
    np.savez_compressed(
        run_dir / f"{split_name}_activations.npz",
        h=hidden.astype(np.float32),
        full_state=data["full_state"].astype(np.int64),
        quotient_state=data["quotient_state"].astype(np.int64),
        nuisance=data["nuisance"].astype(np.int64),
        pair_id=data["pair_id"].astype(np.int64),
        sequence_id=data["sequence_id"].astype(np.int64),
        position=np.broadcast_to(
            np.arange(n_positions, dtype=np.int64),
            (n_sequences, n_positions),
        ).copy(),
        model_seed=np.full(
            (n_sequences, n_positions), 11, dtype=np.int64
        ),
    )


save_activation_export("train", train_data, train_h)
save_activation_export("validation", validation_data, validation_h)


# ---------- Fit fixed-capacity probes on train; evaluate on validation ----------

# The primary test is whether nuisance/sign can be decoded from h alone.
# The oracle-conditioned probe additionally receives the true quotient label.
# Validation data is never used to fit or select a probe.

def flatten_probe_data(data, hidden, include_true_quotient=False):
    n_sequences, n_positions, d_model = hidden.shape
    features = hidden.reshape(-1, d_model)
    labels = data["nuisance"].reshape(-1).astype(np.int64)

    if include_true_quotient:
        quotient = data["quotient_state"]
        quotient_one_hot = np.eye(4, dtype=np.float32)[quotient]
        quotient_one_hot = quotient_one_hot.reshape(-1, 4)
        features = np.concatenate(
            [features, quotient_one_hot],
            axis=1,
        )

    return features, labels


def build_probe(kind, seed):
    if kind == "linear":
        classifier = LogisticRegression(
            C=1.0,
            solver="lbfgs",
            max_iter=1000,
            random_state=seed,
        )
    elif kind == "nonlinear":
        classifier = MLPClassifier(
            hidden_layer_sizes=(32,),
            activation="relu",
            solver="adam",
            alpha=1e-3,
            batch_size=4096,
            learning_rate_init=1e-3,
            max_iter=100,
            early_stopping=False,
            random_state=seed,
        )
    else:
        raise ValueError(kind)

    return Pipeline([
        ("scale", StandardScaler()),
        ("classifier", classifier),
    ])


def position_accuracies(predictions, labels):
    n_sequences, n_positions = labels.shape
    predictions = predictions.reshape(n_sequences, n_positions)
    return [
        float(accuracy_score(labels[:, pos], predictions[:, pos]))
        for pos in range(n_positions)
    ]


probe_results = {}

for conditioning, use_true_quotient in [
    ("hidden_only", False),
    ("hidden_plus_true_quotient", True),
]:
    x_train, y_train = flatten_probe_data(
        train_data,
        train_h,
        include_true_quotient=use_true_quotient,
    )
    x_validation, y_validation = flatten_probe_data(
        validation_data,
        validation_h,
        include_true_quotient=use_true_quotient,
    )

    for probe_kind in ("linear", "nonlinear"):
        probe_name = f"{conditioning}_{probe_kind}"
        probe = build_probe(
            probe_kind,
            seed=1100 + len(probe_results),
        )
        probe.fit(x_train, y_train)

        train_predictions = probe.predict(x_train)
        validation_predictions = probe.predict(x_validation)

        probe_results[probe_name] = {
            "train_accuracy": float(
                accuracy_score(y_train, train_predictions)
            ),
            "validation_accuracy_all_positions": float(
                accuracy_score(y_validation, validation_predictions)
            ),
            "validation_accuracy_by_position": position_accuracies(
                validation_predictions,
                validation_data["nuisance"],
            ),
            "n_train_tokens": int(len(y_train)),
            "n_validation_tokens": int(len(y_validation)),
            "probe_hyperparameters": (
                {
                    "C": 1.0,
                    "solver": "lbfgs",
                    "max_iter": 1000,
                }
                if probe_kind == "linear"
                else {
                    "hidden_layer_sizes": [32],
                    "alpha": 1e-3,
                    "batch_size": 4096,
                    "max_iter": 100,
                    "early_stopping": False,
                }
            ),
        }

        print(f"\n{probe_name}")
        print(
            "  train accuracy:",
            f"{probe_results[probe_name]['train_accuracy']:.3%}",
        )
        print(
            "  validation accuracy, all positions:",
            f"{probe_results[probe_name]['validation_accuracy_all_positions']:.3%}",
        )
        print(
            "  validation accuracy by position:",
            [
                round(value, 3)
                for value in probe_results[probe_name][
                    "validation_accuracy_by_position"
                ]
            ],
        )


# Save the original conditional-head validation diagnostics for comparison.
best_validation = conditional_summary["best_validation"]

results = {
    "experiment": "q8_conditional_head_sign_probe",
    "model_seed": 11,
    "checkpoint_epoch": checkpoint.get("epoch"),
    "representation": "final_norm hidden state immediately before output heads",
    "train_sha256": expected_hashes["train"],
    "validation_sha256": expected_hashes["validation"],
    "test_evaluated": False,
    "original_conditional_head_validation": {
        "full_token_accuracy": best_validation["full_token_accuracy"],
        "quotient_token_accuracy": best_validation[
            "quotient_token_accuracy"
        ],
        "sign_accuracy_given_true_quotient": best_validation[
            "sign_accuracy_given_true_quotient"
        ],
        "sign_accuracy_given_predicted_quotient": best_validation[
            "sign_accuracy_given_predicted_quotient"
        ],
    },
    "probe_results": probe_results,
}

with (run_dir / "probe_metrics.json").open(
    "w", encoding="utf-8"
) as output_file:
    json.dump(results, output_file, indent=2)

archive_path = shutil.make_archive(
    str(run_dir),
    "zip",
    root_dir=run_dir,
)

print("\nDONE")
print("Probe output directory:", run_dir)
print("Download this file from Kaggle:", archive_path)
print("Test split was not opened or evaluated.")